In [65]:
%run udf_game_team_stats 

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 25, Finished, Available, Finished, True)

+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|game_id|team_id|home_or_away|  won|settled_in|head_coach|goals|shots|hits|penalty_in_minutes|power_play_opportunities|power_play_goals|face_off_win_percentage|giveaways|takeaways|blocked|start_rink_side|game_result|
+-------+-------+------------+-----+----------+----------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|      1|    101|        home| true|       REG|   Coach A|    4|   30|  10|                 6|                       4|               1|                   52.5|        5|        8|     12|           left|        win|
|      1|    102|        away|false|       REG|   Coach B|    2|   25|  12|                 8|                       3|             

INFO: Starting relational checks on shots, goals, power_play_goals, power_play_opportunities
INFO: 0 record goals < shots, check passed


INFO: 0 record power_play_goals < goals, check passed
INFO: 0 record showing power_play_opportunities = 0 & goals > 0, check passed
INFO: Starting relational checks on shots, goals, power_play_goals, power_play_opportunities
INFO: Checking won column validity


INFO: Pass - There are 0 false win/lose count and 0 ties scenarios.
INFO: Checking won column validity


INFO: Creating new column game_result
INFO: Successfully created new column game_result


INFO: Checking game_result column validity
INFO: Pass - There are 0 null in game_result.


INFO: Checking game_result column validity


# <u>**Extract**</u>

In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
display(notebookutils.fs.ls("Files/nhl_raw_20260901"))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4725fa4-ac6a-4a79-9370-5dcb031d724e)

-CSVs are under '/lakehouse/default/Files/nhl_raw_20260901'.
-Create a dictionary of Spark DataFrames.

In [3]:
csv_files = [
    "game.csv",
    "team_info.csv",
    "player_info.csv",
    "game_teams_stats.csv",
    "game_skater_stats.csv",
    "game_goalie_stats.csv",
    "game_plays.csv",
    "game_plays_players.csv",
    "game_shifts.csv",
    "game_goals.csv",
    "game_penalties.csv",
    "game_officials.csv",
    "game_scratches.csv"
]

tables = {}

for file in csv_files:
    table_name = file.replace(".csv", "")
    
    tables[table_name] = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"Files/nhl_raw_20260901/{file}")
    )

print("All 13 CSV files loaded.")

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 27, Finished, Available, Finished, False)

All 13 CSV files loaded.


# <u>**Check the tables and for duplicates**</u>

In [4]:
for name, df in tables.items():
    print(f"{name}: {df.count():,} rows × {len(df.columns)} columns")

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 28, Finished, Available, Finished, False)

game: 26,305 rows × 15 columns
team_info: 33 rows × 6 columns
player_info: 3,925 rows × 12 columns
game_teams_stats: 52,610 rows × 17 columns
game_skater_stats: 945,830 rows × 22 columns
game_goalie_stats: 56,656 rows × 19 columns
game_plays: 5,050,529 rows × 18 columns


game_plays_players: 7,586,604 rows × 4 columns
game_shifts: 11,882,735 rows × 5 columns
game_goals: 148,992 rows × 4 columns
game_penalties: 247,828 rows × 3 columns
game_officials: 105,853 rows × 3 columns
game_scratches: 137,333 rows × 3 columns


**Primary Key and Duplicate Checks**

## **Check 'game'**

In [5]:
game_df = tables["game"]

display(game_df.limit(20))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2cb748e1-8eb9-4db6-ba49-832b53f3a907)

In [6]:
game_df.printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 30, Finished, Available, Finished, False)

root
 |-- game_id: integer (nullable = true)
 |-- season: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- date_time_GMT: timestamp (nullable = true)
 |-- away_team_id: integer (nullable = true)
 |-- home_team_id: integer (nullable = true)
 |-- away_goals: integer (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- outcome: string (nullable = true)
 |-- home_rink_side_start: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- venue_link: string (nullable = true)
 |-- venue_time_zone_id: string (nullable = true)
 |-- venue_time_zone_offset: integer (nullable = true)
 |-- venue_time_zone_tz: string (nullable = true)



Define a function for missing values

In [7]:
from pyspark.sql.functions import col, sum, when

def missing_values(df):
    return df.select([
        sum(
            when(col(c).isNull() | (col(c) == ""), 1)
            .otherwise(0)
        ).alias(c)
        for c in df.columns
    ])

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 31, Finished, Available, Finished, False)

Check missing values for game table

In [8]:
display(missing_values(game_df))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ed606524-3cc8-4a64-b54a-68bc35572976)

Check duplicate game ids

In [9]:
total_games = game_df.count()

unique_game_ids = (
    game_df
    .select("game_id")
    .distinct()
    .count()
)

print("Total rows:", total_games)
print("Unique game IDs:", unique_game_ids)
print("Duplicate game IDs:", total_games - unique_game_ids)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 33, Finished, Available, Finished, False)

Total rows: 26305
Unique game IDs: 23735
Duplicate game IDs: 2570


Number of times each game_id appears

In [10]:
from pyspark.sql.functions import count

duplicate_games = (
    game_df
    .groupBy("game_id")
    .agg(count("*").alias("row_count"))
    .filter(col("row_count") > 1)
    .orderBy(col("row_count").desc())
)

display(duplicate_games)


StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 364cf175-8b16-4ef9-971e-fe4a49dde726)

Inspect actual duplicate records

In [163]:
duplicate_ids = (
    duplicate_games
    .select("game_id")
    .limit(10)
)

display(
    game_df
    .join(duplicate_ids, on="game_id", how="inner")
    .orderBy("game_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6ef5f94f-3559-4249-94da-dcd4fa144f59)

Remove the exact duplicates

In [164]:
game_clean_df = game_df.dropDuplicates()

print("Original rows:", game_df.count())
print("Cleaned rows:", game_clean_df.count())
print(
    "Unique game IDs:",
    game_clean_df.select("game_id").distinct().count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 36, Finished, Available, Finished, False)

Original rows: 26305
Cleaned rows: 23735
Unique game IDs: 23735


Verify

In [165]:
total_games = game_clean_df.count()
unique_game_ids = game_clean_df.select("game_id").distinct().count()

print("Total cleaned rows:", total_games)
print("Unique game IDs:", unique_game_ids)
print("Duplicate game IDs:", total_games - unique_game_ids)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 37, Finished, Available, Finished, False)

Total cleaned rows: 23735
Unique game IDs: 23735
Duplicate game IDs: 0


## **Move to team_info**

In [166]:
team_df = tables["team_info"]

display(team_df)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ac4afce-e85e-46df-a6fd-62afe02675b2)

In [167]:
print("Total teams:", team_df.count())
print("Unique team IDs:", team_df.select("team_id").distinct().count())

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 39, Finished, Available, Finished, False)

Total teams: 33
Unique team IDs: 33


Validate the team relationship

In [168]:
team_df = tables["team_info"]

invalid_home_teams = (
    game_df
    .select("home_team_id")
    .distinct()
    .join(
        team_df.select("team_id").distinct(),
        col("home_team_id") == col("team_id"),
        "left_anti"
    )
)

invalid_away_teams = (
    game_df
    .select("away_team_id")
    .distinct()
    .join(
        team_df.select("team_id").distinct(),
        col("away_team_id") == col("team_id"),
        "left_anti"
    )
)

print("Invalid home team IDs:")
display(invalid_home_teams)

print("Invalid away team IDs:")
display(invalid_away_teams)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 40, Finished, Available, Finished, False)

Invalid home team IDs:


SynapseWidget(Synapse.DataFrame, 59689c07-bf3b-4ef7-8b63-620ee4487216)

Invalid away team IDs:


SynapseWidget(Synapse.DataFrame, a06ee6cb-9b2b-4d98-bc32-fd346cd7b366)

In [169]:
# Investigate games involving the invalid team IDs

invalid_team_ids = [87, 88, 89, 90]

display(
    game_clean_df
    .filter(
        col("home_team_id").isin(invalid_team_ids) |
        col("away_team_id").isin(invalid_team_ids)
    )
    .select(
        "game_id",
        "season",
        "date_time_GMT",
        "away_team_id",
        "home_team_id",
        "away_goals",
        "home_goals",
        "outcome"
    )
    .orderBy("game_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d5c516d-aa2b-4d20-8457-e3fef11de7e3)

In [170]:
# Count how many games involve each invalid team ID

display(
    game_clean_df
    .select("home_team_id")
    .union(
        game_clean_df.select("away_team_id")
    )
    .filter(col("home_team_id").isin(invalid_team_ids))
    .groupBy("home_team_id")
    .count()
    .orderBy("home_team_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e074201b-7b7d-46de-8373-ab7caa313802)

These are NHL All-Star games, where the special team IDs 87–90 represent the temporary All-Star teams, rather than regular NHL franchises. So these are legitimate records, not corrupt game data.

In [171]:
display(team_df.orderBy("team_id"))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3edc2b09-598c-4f60-bd7d-5af0ddeb6a1f)

In [172]:
game_teams_stats_df = tables["game_teams_stats"]

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 44, Finished, Available, Finished, False)

In [173]:
display(
    game_teams_stats_df
    .filter(col("team_id").isin([87, 88, 89, 90]))
    .select(
        "game_id",
        "team_id",
        "HoA",
        "head_coach",
        "goals",
        "shots",
        "won",
        "settled_in"
    )
    .orderBy("game_id", "team_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e367b737-1011-446f-b0bc-31eb1e642087)

Validate team_info

In [174]:
team_df = tables["team_info"]

team_df.printSchema()
display(team_df.limit(10))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 46, Finished, Available, Finished, False)

root
 |-- team_id: integer (nullable = true)
 |-- franchiseId: integer (nullable = true)
 |-- shortName: string (nullable = true)
 |-- teamName: string (nullable = true)
 |-- abbreviation: string (nullable = true)
 |-- link: string (nullable = true)



SynapseWidget(Synapse.DataFrame, a79d39e1-1304-423e-810b-42d2a44bb8b7)

Check duplicate team records

In [175]:
team_duplicate_rows = (
    team_df
    .groupBy("team_id")
    .count()
    .filter(col("count") > 1)
)

display(team_duplicate_rows)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f7870ef-9bad-4fd5-8a1a-820dd68ebcbc)

## **Move to player_info**

In [176]:
# Create the DataFrame:
player_df = tables["player_info"]

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 48, Finished, Available, Finished, False)

In [177]:
player_df.printSchema()
display(player_df.limit(10))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 49, Finished, Available, Finished, False)

root
 |-- player_id: integer (nullable = true)
 |-- firstName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- birthCity: string (nullable = true)
 |-- primaryPosition: string (nullable = true)
 |-- birthDate: timestamp (nullable = true)
 |-- birthStateProvince: string (nullable = true)
 |-- height: string (nullable = true)
 |-- height_cm: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- shootsCatches: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 687091ac-1c08-4601-91bf-6a61f019a08b)

Check the number of players

In [178]:
total_players = player_df.count()
unique_players = player_df.select("player_id").distinct().count()

print("Total players:", total_players)
print("Unique player IDs:", unique_players)
print("Duplicate player IDs:", total_players - unique_players)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 50, Finished, Available, Finished, False)

Total players: 3925
Unique player IDs: 3925
Duplicate player IDs: 0


Check any missing player information

In [179]:
display(missing_values(player_df))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 011d2740-ccb6-4f54-afb9-237118b89449)

## **Move to game_teams_stats**

Load and inspect the table

In [180]:
game_teams_stats_df = tables["game_teams_stats"]

game_teams_stats_df.printSchema()
display(game_teams_stats_df.limit(10))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 52, Finished, Available, Finished, False)

root
 |-- game_id: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- HoA: string (nullable = true)
 |-- won: boolean (nullable = true)
 |-- settled_in: string (nullable = true)
 |-- head_coach: string (nullable = true)
 |-- goals: string (nullable = true)
 |-- shots: string (nullable = true)
 |-- hits: string (nullable = true)
 |-- pim: string (nullable = true)
 |-- powerPlayOpportunities: string (nullable = true)
 |-- powerPlayGoals: string (nullable = true)
 |-- faceOffWinPercentage: string (nullable = true)
 |-- giveaways: string (nullable = true)
 |-- takeaways: string (nullable = true)
 |-- blocked: string (nullable = true)
 |-- startRinkSide: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 05486e15-ad52-4a4e-bc49-528cf337c9c9)

Check the row count

In [181]:
print("Total rows:", game_teams_stats_df.count())
print("Expected rows:", game_clean_df.count() * 2)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 53, Finished, Available, Finished, False)

Total rows: 52610
Expected rows: 47470


In [182]:
total_rows = game_teams_stats_df.count()

unique_game_team = (
    game_teams_stats_df
    .select("game_id", "team_id")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Unique (game_id, team_id):", unique_game_team)
print("Duplicate (game_id, team_id):", total_rows - unique_game_team)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 54, Finished, Available, Finished, False)

Total rows: 52610
Unique (game_id, team_id): 47470
Duplicate (game_id, team_id): 5140


In [183]:
# Find duplicated (game_id, team_id) combinations

duplicate_game_teams = (
    game_teams_stats_df
    .groupBy("game_id", "team_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

display(duplicate_game_teams.limit(20))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f0e799d8-307d-43e9-ace5-6efda5475020)

In [184]:
duplicate_pairs = duplicate_game_teams.select(
    "game_id", "team_id"
).limit(10)

display(
    game_teams_stats_df
    .join(
        duplicate_pairs,
        on=["game_id", "team_id"],
        how="inner"
    )
    .orderBy("game_id", "team_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e75446bf-e312-4fbb-a633-a2c04c4ce730)

In [185]:
game_teams_stats_clean_df = game_teams_stats_df.dropDuplicates()

print("Original rows:", game_teams_stats_df.count())
print("Cleaned rows:", game_teams_stats_clean_df.count())
print(
    "Unique (game_id, team_id):",
    game_teams_stats_clean_df
    .select("game_id", "team_id")
    .distinct()
    .count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 57, Finished, Available, Finished, False)

Original rows: 52610
Cleaned rows: 47470
Unique (game_id, team_id): 47470


In [186]:
duplicate_count = (
    game_teams_stats_clean_df
    .groupBy("game_id", "team_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate (game_id, team_id) combinations:", duplicate_count)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 58, Finished, Available, Finished, False)

Duplicate (game_id, team_id) combinations: 0


In [187]:
invalid_games = (
    game_teams_stats_clean_df
    .select("game_id")
    .distinct()
    .join(
        game_clean_df.select("game_id").distinct(),
        on="game_id",
        how="left_anti"
    )
)

print("Invalid game IDs:", invalid_games.count())
display(invalid_games)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 59, Finished, Available, Finished, False)

Invalid game IDs: 0


SynapseWidget(Synapse.DataFrame, 04e413ea-393c-4b15-ada8-796074d0a60c)

In [188]:
invalid_teams = (
    game_teams_stats_clean_df
    .select("team_id")
    .distinct()
    .join(
        team_df.select("team_id").distinct(),
        on="team_id",
        how="left_anti"
    )
)

print("Invalid team IDs:", invalid_teams.count())
display(invalid_teams)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 60, Finished, Available, Finished, False)

Invalid team IDs: 4


SynapseWidget(Synapse.DataFrame, a310dcdb-41d8-4dc3-adee-e1034c9b541f)

## **Move to game_skater_stats**

Load and Inspect

In [189]:
game_skater_stats_df = tables["game_skater_stats"]

game_skater_stats_df.printSchema()
display(game_skater_stats_df.limit(10))


StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 61, Finished, Available, Finished, False)

root
 |-- game_id: integer (nullable = true)
 |-- player_id: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- timeOnIce: integer (nullable = true)
 |-- assists: integer (nullable = true)
 |-- goals: integer (nullable = true)
 |-- shots: integer (nullable = true)
 |-- hits: string (nullable = true)
 |-- powerPlayGoals: integer (nullable = true)
 |-- powerPlayAssists: integer (nullable = true)
 |-- penaltyMinutes: integer (nullable = true)
 |-- faceOffWins: integer (nullable = true)
 |-- faceoffTaken: integer (nullable = true)
 |-- takeaways: string (nullable = true)
 |-- giveaways: string (nullable = true)
 |-- shortHandedGoals: integer (nullable = true)
 |-- shortHandedAssists: integer (nullable = true)
 |-- blocked: string (nullable = true)
 |-- plusMinus: integer (nullable = true)
 |-- evenTimeOnIce: integer (nullable = true)
 |-- shortHandedTimeOnIce: integer (nullable = true)
 |-- powerPlayTimeOnIce: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, 09f45b0c-ea28-4242-8c7d-c885180c9d92)

In [190]:
print("Total rows:", game_skater_stats_df.count())

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 62, Finished, Available, Finished, False)

Total rows: 945830


Test (game_id, player_id) uniqueness

In [191]:
total_rows = game_skater_stats_df.count()

unique_game_player = (
    game_skater_stats_df
    .select("game_id", "player_id")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Unique (game_id, player_id):", unique_game_player)
print("Duplicate (game_id, player_id):", total_rows - unique_game_player)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 63, Finished, Available, Finished, False)

Total rows: 945830
Unique (game_id, player_id): 853404
Duplicate (game_id, player_id): 92426


Validate game_id

In [192]:
invalid_games_skater = (
    game_skater_stats_df
    .select("game_id")
    .distinct()
    .join(
        game_clean_df.select("game_id").distinct(),
        on="game_id",
        how="left_anti"
    )
)

print("Invalid game IDs:", invalid_games_skater.count())
display(invalid_games_skater)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 64, Finished, Available, Finished, False)

Invalid game IDs: 0


SynapseWidget(Synapse.DataFrame, 830d50ec-46e8-4285-9914-25cc5104af19)

Validate player_id

In [193]:
invalid_players_skater = (
    game_skater_stats_df
    .select("player_id")
    .distinct()
    .join(
        player_df.select("player_id").distinct(),
        on="player_id",
        how="left_anti"
    )
)

print("Invalid player IDs:", invalid_players_skater.count())
display(invalid_players_skater)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 65, Finished, Available, Finished, False)

Invalid player IDs: 0


SynapseWidget(Synapse.DataFrame, daad881f-7ba3-4934-9e48-91d8819a5968)

Validate team_id

In [194]:
invalid_teams_skater = (
    game_skater_stats_df
    .select("team_id")
    .distinct()
    .join(
        team_df.select("team_id").distinct(),
        on="team_id",
        how="left_anti"
    )
)

print("Invalid team IDs:", invalid_teams_skater.count())
display(invalid_teams_skater)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 66, Finished, Available, Finished, False)

Invalid team IDs: 4


SynapseWidget(Synapse.DataFrame, bbf59821-cd4f-48f1-aaf7-1da4ba9be94e)

Check the duplicates

In [195]:
duplicate_skater_pairs = (
    game_skater_stats_df
    .groupBy("game_id", "player_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

display(duplicate_skater_pairs.limit(20))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 67, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e047f1d0-e92c-456e-8d6e-54649d7fa412)

In [196]:
duplicate_skater_sample = (
    duplicate_skater_pairs
    .select("game_id", "player_id")
    .limit(10)
)

display(
    game_skater_stats_df
    .join(
        duplicate_skater_sample,
        on=["game_id", "player_id"],
        how="inner"
    )
    .orderBy("game_id", "player_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 68, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c9107bc-8f26-4ef7-9116-fb4ac39b0960)

Check the extra 4 team IDs in this data

In [197]:
invalid_teams_skater = (
    game_skater_stats_df
    .select("team_id")
    .distinct()
    .join(
        team_df.select("team_id").distinct(),
        on="team_id",
        how="left_anti"
    )
)

display(invalid_teams_skater)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 69, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0a6a9acc-e43c-4894-84d4-eae39c3202fd)

In [198]:
display(
    game_skater_stats_df
    .join(
        invalid_teams_skater,
        on="team_id",
        how="inner"
    )
    .select(
        "game_id",
        "player_id",
        "team_id",
        "timeOnIce",
        "goals",
        "shots"
    )
    .orderBy("game_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 70, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05fbe7ff-8af9-490a-a8d6-39d66b9a69e6)

Remove the duplicates

In [199]:
game_skater_stats_clean_df = game_skater_stats_df.dropDuplicates()

print("Original rows:", game_skater_stats_df.count())
print("Cleaned rows:", game_skater_stats_clean_df.count())

unique_game_player = (
    game_skater_stats_clean_df
    .select("game_id", "player_id")
    .distinct()
    .count()
)

print("Unique (game_id, player_id):", unique_game_player)
print(
    "Duplicate (game_id, player_id):",
    game_skater_stats_clean_df.count() - unique_game_player
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 71, Finished, Available, Finished, False)

Original rows: 945830
Cleaned rows: 853404
Unique (game_id, player_id): 853404
Duplicate (game_id, player_id): 0


## **Move to game_goalie_stats**

In [200]:
game_goalie_stats_df = tables["game_goalie_stats"]

game_goalie_stats_df.printSchema()
display(game_goalie_stats_df.limit(10))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 72, Finished, Available, Finished, False)

root
 |-- game_id: integer (nullable = true)
 |-- player_id: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- timeOnIce: integer (nullable = true)
 |-- assists: integer (nullable = true)
 |-- goals: integer (nullable = true)
 |-- pim: integer (nullable = true)
 |-- shots: integer (nullable = true)
 |-- saves: integer (nullable = true)
 |-- powerPlaySaves: integer (nullable = true)
 |-- shortHandedSaves: integer (nullable = true)
 |-- evenSaves: integer (nullable = true)
 |-- shortHandedShotsAgainst: integer (nullable = true)
 |-- evenShotsAgainst: integer (nullable = true)
 |-- powerPlayShotsAgainst: integer (nullable = true)
 |-- decision: string (nullable = true)
 |-- savePercentage: string (nullable = true)
 |-- powerPlaySavePercentage: string (nullable = true)
 |-- evenStrengthSavePercentage: string (nullable = true)



SynapseWidget(Synapse.DataFrame, ea002be0-f303-4b97-bb50-834760642964)

In [201]:
print("Total rows:", game_goalie_stats_df.count())

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 73, Finished, Available, Finished, False)

Total rows: 56656


In [202]:
total_rows = game_goalie_stats_df.count()

unique_game_player = (
    game_goalie_stats_df
    .select("game_id", "player_id")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Unique (game_id, player_id):", unique_game_player)
print(
    "Duplicate (game_id, player_id):",
    total_rows - unique_game_player
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 74, Finished, Available, Finished, False)

Total rows: 56656
Unique (game_id, player_id): 51163
Duplicate (game_id, player_id): 5493


Validate

In [203]:
invalid_games_goalie = (
    game_goalie_stats_df
    .select("game_id")
    .distinct()
    .join(
        game_clean_df.select("game_id").distinct(),
        on="game_id",
        how="left_anti"
    )
)

print("Invalid game IDs:", invalid_games_goalie.count())
display(invalid_games_goalie)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 75, Finished, Available, Finished, False)

Invalid game IDs: 0


SynapseWidget(Synapse.DataFrame, c3560f7e-91cc-4054-bedf-b59c2d23266c)

In [204]:
invalid_players_goalie = (
    game_goalie_stats_df
    .select("player_id")
    .distinct()
    .join(
        player_df.select("player_id").distinct(),
        on="player_id",
        how="left_anti"
    )
)

print("Invalid player IDs:", invalid_players_goalie.count())
display(invalid_players_goalie)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 76, Finished, Available, Finished, False)

Invalid player IDs: 0


SynapseWidget(Synapse.DataFrame, fef80e88-9c68-4b5c-b024-7c0cdbd07d29)

In [205]:
invalid_teams_goalie = (
    game_goalie_stats_df
    .select("team_id")
    .distinct()
    .join(
        team_df.select("team_id").distinct(),
        on="team_id",
        how="left_anti"
    )
)

print("Invalid team IDs:", invalid_teams_goalie.count())
display(invalid_teams_goalie)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 77, Finished, Available, Finished, False)

Invalid team IDs: 4


SynapseWidget(Synapse.DataFrame, 65234f73-b901-4484-8f0f-b67ad5bb4db4)

Check the duplicates

In [206]:
duplicate_goalie_pairs = (
    game_goalie_stats_df
    .groupBy("game_id", "player_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

display(duplicate_goalie_pairs.limit(20))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 78, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 25f35cf3-9ef5-4375-b8db-0d4bb6379f7b)

In [207]:
duplicate_goalie_sample = (
    duplicate_goalie_pairs
    .select("game_id", "player_id")
    .limit(10)
)

display(
    game_goalie_stats_df
    .join(
        duplicate_goalie_sample,
        on=["game_id", "player_id"],
        how="inner"
    )
    .orderBy("game_id", "player_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 79, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e2bc793c-157c-424c-8e3c-e241410b8ba7)

Clean the table

In [208]:
game_goalie_stats_clean_df = game_goalie_stats_df.dropDuplicates()

print("Original rows:", game_goalie_stats_df.count())
print("Cleaned rows:", game_goalie_stats_clean_df.count())

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 80, Finished, Available, Finished, False)

Original rows: 56656
Cleaned rows: 51163


Recheck

In [209]:
unique_game_player = (
    game_goalie_stats_clean_df
    .select("game_id", "player_id")
    .distinct()
    .count()
)

total_rows = game_goalie_stats_clean_df.count()

print("Total rows:", total_rows)
print("Unique (game_id, player_id):", unique_game_player)
print("Duplicate (game_id, player_id):", total_rows - unique_game_player)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 81, Finished, Available, Finished, False)

Total rows: 51163
Unique (game_id, player_id): 51163
Duplicate (game_id, player_id): 0


## **Move to game_plays**

In [210]:
game_plays_df = tables["game_plays"]

total_plays = game_plays_df.count()
unique_play_ids = game_plays_df.select("play_id").distinct().count()

print("Total rows:", total_plays)
print("Unique play_id:", unique_play_ids)
print("Duplicate play_id:", total_plays - unique_play_ids)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 82, Finished, Available, Finished, False)

Total rows: 5050529
Unique play_id: 4217063
Duplicate play_id: 833466


In [211]:
game_plays_df = tables["game_plays"]

game_plays_df.printSchema()
display(game_plays_df.limit(10))

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 83, Finished, Available, Finished, False)

root
 |-- play_id: string (nullable = true)
 |-- game_id: integer (nullable = true)
 |-- team_id_for: string (nullable = true)
 |-- team_id_against: string (nullable = true)
 |-- event: string (nullable = true)
 |-- secondaryType: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- period: integer (nullable = true)
 |-- periodType: string (nullable = true)
 |-- periodTime: integer (nullable = true)
 |-- periodTimeRemaining: string (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- goals_away: integer (nullable = true)
 |-- goals_home: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- st_x: string (nullable = true)
 |-- st_y: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 5ee8ebb6-a4b7-4c0e-a539-de30f2767a02)

In [212]:
game_plays_df = tables["game_plays"]

total_plays = game_plays_df.count()
unique_play_ids = game_plays_df.select("play_id").distinct().count()

print("Total rows:", total_plays)
print("Unique play_id:", unique_play_ids)
print("Duplicate play_id:", total_plays - unique_play_ids)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 84, Finished, Available, Finished, False)

Total rows: 5050529
Unique play_id: 4217063
Duplicate play_id: 833466


In [213]:
# Remove exact duplicate rows
game_plays_no_exact_dupes_df = game_plays_df.dropDuplicates()

original_count = game_plays_df.count()
cleaned_count = game_plays_no_exact_dupes_df.count()

print("Original rows:", original_count)
print("After removing exact duplicates:", cleaned_count)
print("Exact duplicate rows removed:", original_count - cleaned_count)


StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 85, Finished, Available, Finished, False)

Original rows: 5050529
After removing exact duplicates: 4217063
Exact duplicate rows removed: 833466


In [214]:
unique_play_ids = (
    game_plays_no_exact_dupes_df
    .select("play_id")
    .distinct()
    .count()
)

total_rows = game_plays_no_exact_dupes_df.count()

print("Total rows:", total_rows)
print("Unique play_id:", unique_play_ids)
print("Remaining duplicate play_id:", total_rows - unique_play_ids)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 86, Finished, Available, Finished, False)

Total rows: 4217063
Unique play_id: 4217063
Remaining duplicate play_id: 0


## **Move to game_plays_players**

In [215]:
game_plays_players_df = tables["game_plays_players"]

total_rows = game_plays_players_df.count()

unique_combinations = (
    game_plays_players_df
    .select("play_id", "player_id", "playerType")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Unique (play_id, player_id, playerType):", unique_combinations)
print(
    "Duplicate (play_id, player_id, playerType):",
    total_rows - unique_combinations
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 87, Finished, Available, Finished, False)

Total rows: 7586604
Unique (play_id, player_id, playerType): 6362804
Duplicate (play_id, player_id, playerType): 1223800


In [216]:
game_plays_players_no_exact_dupes_df = (
    game_plays_players_df.dropDuplicates()
)

original_count = game_plays_players_df.count()
cleaned_count = game_plays_players_no_exact_dupes_df.count()

print("Original rows:", original_count)
print("After removing exact duplicates:", cleaned_count)
print("Exact duplicate rows removed:", original_count - cleaned_count)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 88, Finished, Available, Finished, False)

Original rows: 7586604
After removing exact duplicates: 6362804
Exact duplicate rows removed: 1223800


Identifying the combinations

In [217]:
unique_combinations = (
    game_plays_players_no_exact_dupes_df
    .select("play_id", "player_id", "playerType")
    .distinct()
    .count()
)

print("Unique (play_id, player_id, playerType):", unique_combinations)
print(
    "Remaining duplicate combinations:",
    cleaned_count - unique_combinations
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 89, Finished, Available, Finished, False)

Unique (play_id, player_id, playerType): 6362804
Remaining duplicate combinations: 0


In [218]:
unique_play_player = (
    game_plays_players_no_exact_dupes_df
    .select("play_id", "player_id")
    .distinct()
    .count()
)

print("Unique (play_id, player_id):", unique_play_player)
print(
    "Duplicate (play_id, player_id):",
    cleaned_count - unique_play_player
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 90, Finished, Available, Finished, False)

Unique (play_id, player_id): 6362801
Duplicate (play_id, player_id): 3


## **Move to game_shifts**

In [219]:
game_shifts_df = tables["game_shifts"]

total_rows = game_shifts_df.count()

unique_shifts = (
    game_shifts_df
    .select(
        "game_id",
        "player_id",
        "period",
        "shift_start",
        "shift_end"
    )
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Unique shift combinations:", unique_shifts)
print("Duplicate shift combinations:", total_rows - unique_shifts)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 91, Finished, Available, Finished, False)

Total rows: 11882735
Unique shift combinations: 9900705
Duplicate shift combinations: 1982030


Check exact duplicates 

In [220]:
game_shifts_no_exact_dupes_df = game_shifts_df.dropDuplicates()

original_count = game_shifts_df.count()
cleaned_count = game_shifts_no_exact_dupes_df.count()

print("Original rows:", original_count)
print("After removing exact duplicates:", cleaned_count)
print("Exact duplicate rows removed:", original_count - cleaned_count)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 92, Finished, Available, Finished, False)

Original rows: 11882735
After removing exact duplicates: 9900705
Exact duplicate rows removed: 1982030


Recheck the proposed key

In [221]:
unique_shifts = (
    game_shifts_no_exact_dupes_df
    .select(
        "game_id",
        "player_id",
        "period",
        "shift_start",
        "shift_end"
    )
    .distinct()
    .count()
)

print("Unique shift combinations:", unique_shifts)
print(
    "Remaining duplicate combinations:",
    cleaned_count - unique_shifts
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 93, Finished, Available, Finished, False)

Unique shift combinations: 9900705
Remaining duplicate combinations: 0


## **Move to game_goals**

In [222]:
game_goals_df = tables["game_goals"]

total_rows = game_goals_df.count()

unique_play_ids = (
    game_goals_df
    .select("play_id")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Unique play_id:", unique_play_ids)
print("Duplicate play_id:", total_rows - unique_play_ids)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 94, Finished, Available, Finished, False)

Total rows: 148992
Unique play_id: 133345
Duplicate play_id: 15647


In [223]:
game_goals_no_exact_dupes_df = game_goals_df.dropDuplicates()

original_count = game_goals_df.count()
cleaned_count = game_goals_no_exact_dupes_df.count()

print("Original rows:", original_count)
print("After removing exact duplicates:", cleaned_count)
print("Exact duplicate rows removed:", original_count - cleaned_count)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 95, Finished, Available, Finished, False)

Original rows: 148992
After removing exact duplicates: 133345
Exact duplicate rows removed: 15647


In [224]:
unique_play_ids = (
    game_goals_no_exact_dupes_df
    .select("play_id")
    .distinct()
    .count()
)

print("Unique play_id:", unique_play_ids)
print(
    "Remaining duplicate play_id:",
    cleaned_count - unique_play_ids
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 96, Finished, Available, Finished, False)

Unique play_id: 133345
Remaining duplicate play_id: 0


In [225]:
game_goals_clean_df = game_goals_df.dropDuplicates()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 97, Finished, Available, Finished, False)

## **Move to game_penalties**

In [226]:
game_penalties_df = tables["game_penalties"]

total_rows = game_penalties_df.count()

unique_play_ids = (
    game_penalties_df
    .select("play_id")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Unique play_id:", unique_play_ids)
print("Duplicate play_id:", total_rows - unique_play_ids)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 98, Finished, Available, Finished, False)

Total rows: 247828
Unique play_id: 229228
Duplicate play_id: 18600


In [227]:
game_penalties_no_exact_dupes_df = game_penalties_df.dropDuplicates()

original_count = game_penalties_df.count()
cleaned_count = game_penalties_no_exact_dupes_df.count()

print("Original rows:", original_count)
print("After removing exact duplicates:", cleaned_count)
print("Exact duplicate rows removed:", original_count - cleaned_count)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 99, Finished, Available, Finished, False)

Original rows: 247828
After removing exact duplicates: 229228
Exact duplicate rows removed: 18600


In [228]:
unique_play_ids = (
    game_penalties_no_exact_dupes_df
    .select("play_id")
    .distinct()
    .count()
)

print("Unique play_id:", unique_play_ids)
print(
    "Remaining duplicate play_id:",
    cleaned_count - unique_play_ids
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 100, Finished, Available, Finished, False)

Unique play_id: 229228
Remaining duplicate play_id: 0


In [229]:
game_penalties_clean_df = game_penalties_df.dropDuplicates()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 101, Finished, Available, Finished, False)

## **Move to game_officials**

In [230]:
game_officials_df = tables["game_officials"]

total_rows = game_officials_df.count()

unique_combinations = (
    game_officials_df
    .select("game_id", "official_name", "official_type")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print(
    "Unique (game_id, official_name, official_type):",
    unique_combinations
)
print(
    "Duplicate combinations:",
    total_rows - unique_combinations
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 102, Finished, Available, Finished, False)

Total rows: 105853
Unique (game_id, official_name, official_type): 95188
Duplicate combinations: 10665


In [231]:
game_officials_no_exact_dupes_df = game_officials_df.dropDuplicates()

original_count = game_officials_df.count()
cleaned_count = game_officials_no_exact_dupes_df.count()

print("Original rows:", original_count)
print("After removing exact duplicates:", cleaned_count)
print("Exact duplicate rows removed:", original_count - cleaned_count)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 103, Finished, Available, Finished, False)

Original rows: 105853
After removing exact duplicates: 95188
Exact duplicate rows removed: 10665


In [232]:
unique_combinations = (
    game_officials_no_exact_dupes_df
    .select("game_id", "official_name", "official_type")
    .distinct()
    .count()
)

print(
    "Unique (game_id, official_name, official_type):",
    unique_combinations
)
print(
    "Remaining duplicate combinations:",
    cleaned_count - unique_combinations
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 104, Finished, Available, Finished, False)

Unique (game_id, official_name, official_type): 95188
Remaining duplicate combinations: 0


In [233]:
game_officials_clean_df = game_officials_df.dropDuplicates()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 105, Finished, Available, Finished, False)

## **Move to game_scratches**

In [234]:
game_scratches_df = tables["game_scratches"]

total_rows = game_scratches_df.count()

unique_combinations = (
    game_scratches_df
    .select("game_id", "team_id", "player_id")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print(
    "Unique (game_id, team_id, player_id):",
    unique_combinations
)
print(
    "Duplicate combinations:",
    total_rows - unique_combinations
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 106, Finished, Available, Finished, False)

Total rows: 137333
Unique (game_id, team_id, player_id): 119749
Duplicate combinations: 17584


In [235]:
game_scratches_no_exact_dupes_df = game_scratches_df.dropDuplicates()

original_count = game_scratches_df.count()
cleaned_count = game_scratches_no_exact_dupes_df.count()

print("Original rows:", original_count)
print("After removing exact duplicates:", cleaned_count)
print("Exact duplicate rows removed:", original_count - cleaned_count)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 107, Finished, Available, Finished, False)

Original rows: 137333
After removing exact duplicates: 119749
Exact duplicate rows removed: 17584


In [236]:
unique_combinations = (
    game_scratches_no_exact_dupes_df
    .select("game_id", "team_id", "player_id")
    .distinct()
    .count()
)

print(
    "Unique (game_id, team_id, player_id):",
    unique_combinations
)

print(
    "Remaining duplicate combinations:",
    cleaned_count - unique_combinations
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 108, Finished, Available, Finished, False)

Unique (game_id, team_id, player_id): 119749
Remaining duplicate combinations: 0


# <u>**Transform**</u>

## **Adding All-star team ID into team_info**

Step 1: Handle All-Star team IDs. 87, 88, 89, 90

In [237]:
special_team_ids = [87, 88, 89, 90]

display(
    game_clean_df
    .filter(
        col("home_team_id").isin(special_team_ids) |
        col("away_team_id").isin(special_team_ids)
    )
    .select(
        "game_id",
        "season",
        "date_time_GMT",
        "away_team_id",
        "home_team_id",
        "away_goals",
        "home_goals",
        "outcome",
        "venue"
    )
    .orderBy("game_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 109, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 92441fde-8357-474c-a609-bcda420581b7)

Step 1(A1)-Create team_info_clean_df

In [238]:
tables["team_info"]

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 110, Finished, Available, Finished, False)

DataFrame[team_id: int, franchiseId: int, shortName: string, teamName: string, abbreviation: string, link: string]

In [239]:
from pyspark.sql.functions import col

# Get team_info DataFrame from the tables dictionary
team_info_df = tables["team_info"]

# Remove exact duplicate records
team_info_clean_df = team_info_df.dropDuplicates()

# Special/All-Star team IDs found in the game data
special_team_rows = [
    (87, None, None, None, None, None),
    (88, None, None, None, None, None),
    (89, None, None, None, None, None),
    (90, None, None, None, None, None)
]

# Create DataFrame using the same schema as team_info
special_team_df = spark.createDataFrame(
    special_team_rows,
    schema=team_info_clean_df.schema
)

# Add the special team IDs
team_info_clean_df = (
    team_info_clean_df
    .unionByName(special_team_df)
    .dropDuplicates(["team_id"])
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 111, Finished, Available, Finished, False)

Step 1(A2)-Create the result and Compare with Original

In [240]:
print("Original team_info rows:", team_info_df.count())
print("Cleaned team_info rows:", team_info_clean_df.count())

display(
    team_info_clean_df
    .filter(col("team_id").isin([87, 88, 89, 90]))
    .orderBy("team_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 112, Finished, Available, Finished, False)

Original team_info rows: 33
Cleaned team_info rows: 37


SynapseWidget(Synapse.DataFrame, 13286fba-6a0a-4973-a065-5e4841f57ea1)

Step 1(A3)-Check on game_clean_df

In [241]:
display(
    game_clean_df
    .filter(
        (col("home_team_id").isin([87, 88, 89, 90])) |
        (col("away_team_id").isin([87, 88, 89, 90]))
    )
    .select(
        "game_id",
        "season",
        "date_time_GMT",
        "away_team_id",
        "home_team_id",
        "away_goals",
        "home_goals",
        "outcome"
    )
    .orderBy("game_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 113, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 706f7e6b-f975-41bb-bf0a-d28bacaed26f)

Step 1(A4)-Verify the team IDs

In [242]:
# Check that all team IDs used in game are present in team_info_clean_df

game_team_ids = (
    game_clean_df
    .select(col("home_team_id").alias("team_id"))
    .union(
        game_clean_df.select(col("away_team_id").alias("team_id"))
    )
    .distinct()
)

missing_team_ids = (
    game_team_ids
    .join(
        team_info_clean_df.select("team_id"),
        on="team_id",
        how="left_anti"
    )
)

print("Team IDs used in games:", game_team_ids.count())
print("Missing team IDs:", missing_team_ids.count())

display(missing_team_ids)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 212, Finished, Available, Finished, False)

Team IDs used in games: 37
Missing team IDs: 0


SynapseWidget(Synapse.DataFrame, caa44285-26de-468a-b9f6-799b48d1bbc9)

Fully validated

## **Create all cleaned DataFrames and validate all PK → FK relationships**

Step 2 — Create all cleaned DataFrames and drop duplicates

In [243]:
game_clean_df = tables["game"].dropDuplicates()

# Use the team_info_clean_df we created in Step A
team_info_clean_df = team_info_clean_df

player_info_clean_df = tables["player_info"].dropDuplicates()

game_teams_stats_clean_df = tables["game_teams_stats"].dropDuplicates()

game_skater_stats_clean_df = tables["game_skater_stats"].dropDuplicates()

game_goalie_stats_clean_df = tables["game_goalie_stats"].dropDuplicates()

game_plays_clean_df = tables["game_plays"].dropDuplicates()

game_plays_players_clean_df = tables["game_plays_players"].dropDuplicates()

game_shifts_clean_df = tables["game_shifts"].dropDuplicates()

game_goals_clean_df = tables["game_goals"].dropDuplicates()

game_penalties_clean_df = tables["game_penalties"].dropDuplicates()

game_officials_clean_df = tables["game_officials"].dropDuplicates()

game_scratches_clean_df = tables["game_scratches"].dropDuplicates()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 213, Finished, Available, Finished, False)

Verify the row counts.

In [244]:
clean_tables = {
    "game": game_clean_df,
    "team_info": team_info_clean_df,
    "player_info": player_info_clean_df,
    "game_teams_stats": game_teams_stats_clean_df,
    "game_skater_stats": game_skater_stats_clean_df,
    "game_goalie_stats": game_goalie_stats_clean_df,
    "game_plays": game_plays_clean_df,
    "game_plays_players": game_plays_players_clean_df,
    "game_shifts": game_shifts_clean_df,
    "game_goals": game_goals_clean_df,
    "game_penalties": game_penalties_clean_df,
    "game_officials": game_officials_clean_df,
    "game_scratches": game_scratches_clean_df
}

for name, df in clean_tables.items():
    print(f"{name}: {df.count():,}")

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 214, Finished, Available, Finished, False)

game: 23,735
team_info: 37
player_info: 3,925
game_teams_stats: 47,470
game_skater_stats: 853,404


game_goalie_stats: 51,163


game_plays: 4,217,063


game_plays_players: 6,362,804


game_shifts: 9,900,705
game_goals: 133,345
game_penalties: 229,228
game_officials: 95,188
game_scratches: 119,749


###### Step 3 — Validate all PK → FK relationships

In [245]:
from pyspark.sql.functions import col

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 215, Finished, Available, Finished, False)

In [246]:
# Home team FK
invalid_home_teams = validate_fk(
    game_clean_df,
    ["home_team_id"],
    team_info_clean_df,
    ["team_id"]
)

# Away team FK
invalid_away_teams = validate_fk(
    game_clean_df,
    ["away_team_id"],
    team_info_clean_df,
    ["team_id"]
)

print("Invalid home_team_id:", invalid_home_teams.count())
print("Invalid away_team_id:", invalid_away_teams.count())

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 216, Finished, Available, Finished, False)

Invalid home_team_id: 0
Invalid away_team_id: 0


In [1]:
from pyspark.sql import Row

# Complete FK validation

fk_checks = [
    
    # GAME → TEAM_INFO
    ("game", "home_team_id", "team_info", "team_id"),
    ("game", "away_team_id", "team_info", "team_id"),
    
    # GAME_TEAMS_STATS
    ("game_teams_stats", "game_id", "game", "game_id"),
    ("game_teams_stats", "team_id", "team_info", "team_id"),
    
    # GAME_SKATER_STATS
    ("game_skater_stats", "game_id", "game", "game_id"),
    ("game_skater_stats", "player_id", "player_info", "player_id"),
    ("game_skater_stats", "team_id", "team_info", "team_id"),
    
    # GAME_GOALIE_STATS
    ("game_goalie_stats", "game_id", "game", "game_id"),
    ("game_goalie_stats", "player_id", "player_info", "player_id"),
    ("game_goalie_stats", "team_id", "team_info", "team_id"),
    
    # GAME_PLAYS
    ("game_plays", "game_id", "game", "game_id"),
    ("game_plays", "team_id_for", "team_info", "team_id"),
    ("game_plays", "team_id_against", "team_info", "team_id"),
    
    # GAME_PLAYS_PLAYERS
    ("game_plays_players", "play_id", "game_plays", "play_id"),
    ("game_plays_players", "game_id", "game", "game_id"),
    ("game_plays_players", "player_id", "player_info", "player_id"),
    
    # GAME_SHIFTS
    ("game_shifts", "game_id", "game", "game_id"),
    ("game_shifts", "player_id", "player_info", "player_id"),
    
    # GAME_GOALS
    ("game_goals", "play_id", "game_plays", "play_id"),
    
    # GAME_PENALTIES
    ("game_penalties", "play_id", "game_plays", "play_id"),
    
    # GAME_OFFICIALS
    ("game_officials", "game_id", "game", "game_id"),
    
    # GAME_SCRATCHES
    ("game_scratches", "game_id", "game", "game_id"),
    ("game_scratches", "team_id", "team_info", "team_id"),
    ("game_scratches", "player_id", "player_info", "player_id")
]


# ============================================================
# MAP TABLE NAMES TO CLEAN DATAFRAMES
# ============================================================

clean_df_map = {
    "game": game_clean_df,
    "team_info": team_info_clean_df,
    "player_info": player_info_clean_df,
    "game_teams_stats": game_teams_stats_clean_df,
    "game_skater_stats": game_skater_stats_clean_df,
    "game_goalie_stats": game_goalie_stats_clean_df,
    "game_plays": game_plays_clean_df,
    "game_plays_players": game_plays_players_clean_df,
    "game_shifts": game_shifts_clean_df,
    "game_goals": game_goals_clean_df,
    "game_penalties": game_penalties_clean_df,
    "game_officials": game_officials_clean_df,
    "game_scratches": game_scratches_clean_df
}


# ============================================================
# RUN ALL CHECKS
# ============================================================

results = []

for child_table, child_col, parent_table, parent_col in fk_checks:
    
    invalid_count = fk_invalid_count(
        clean_df_map[child_table],
        child_col,
        clean_df_map[parent_table],
        parent_col
    )
    
    results.append(
        (
            child_table,
            child_col,
            parent_table,
            parent_col,
            invalid_count
        )
    )


# ============================================================
# DISPLAY FK VALIDATION REPORT
# ============================================================

fk_validation_df = spark.createDataFrame(
    results,
    [
        "child_table",
        "foreign_key",
        "parent_table",
        "parent_key",
        "invalid_fk_count"
    ]
)

display(
    fk_validation_df.orderBy(
        "invalid_fk_count",
        "child_table"
    )
)

StatementMeta(, 00e4b3bc-05ec-4df2-a5d3-d64254335f35, 3, Finished, Available, Finished, False)

NameError: name 'game_clean_df' is not defined

###### Inspect and resolve team_ids in game_plays

In [248]:
from pyspark.sql.functions import col

# Invalid team_id_for
invalid_team_for = (
    game_plays_clean_df
    .filter(col("team_id_for").isNotNull())
    .select("team_id_for")
    .dropDuplicates()
    .join(
        team_info_clean_df.select(
            col("team_id").alias("valid_team_id")
        ),
        col("team_id_for") == col("valid_team_id"),
        "left_anti"
    )
)

# Invalid team_id_against
invalid_team_against = (
    game_plays_clean_df
    .filter(col("team_id_against").isNotNull())
    .select("team_id_against")
    .dropDuplicates()
    .join(
        team_info_clean_df.select(
            col("team_id").alias("valid_team_id")
        ),
        col("team_id_against") == col("valid_team_id"),
        "left_anti"
    )
)

print("Invalid team_id_for:")
display(invalid_team_for)

print("Invalid team_id_against:")
display(invalid_team_against)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 218, Finished, Available, Finished, False)

Invalid team_id_for:


SynapseWidget(Synapse.DataFrame, acf6083b-4d4b-47ab-8bf4-41e99cb5dddf)

Invalid team_id_against:


SynapseWidget(Synapse.DataFrame, d9435d74-5492-4ebd-97ca-6194bb437669)

In [249]:
from pyspark.sql.functions import col, when

game_plays_clean_df = (
    game_plays_clean_df
    .withColumn(
        "team_id_for",
        when(col("team_id_for") == "NA", None)
        .otherwise(col("team_id_for"))
    )
    .withColumn(
        "team_id_against",
        when(col("team_id_against") == "NA", None)
        .otherwise(col("team_id_against"))
    )
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 219, Finished, Available, Finished, False)

In [250]:
display(
    game_plays_clean_df
    .filter(
        col("team_id_for").isNull() |
        col("team_id_against").isNull()
    )
    .select(
        "play_id",
        "game_id",
        "team_id_for",
        "team_id_against",
        "event",
        "description"
    )
    .limit(20)
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 220, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8bf296e2-1ff3-4a80-9ec2-a272a1aa45c2)

In [251]:
invalid_team_for = validate_fk(
    game_plays_clean_df.filter(col("team_id_for").isNotNull()),
    ["team_id_for"],
    team_info_clean_df,
    ["team_id"]
)

invalid_team_against = validate_fk(
    game_plays_clean_df.filter(col("team_id_against").isNotNull()),
    ["team_id_against"],
    team_info_clean_df,
    ["team_id"]
)

print("Invalid team_id_for:", invalid_team_for.count())
print("Invalid team_id_against:", invalid_team_against.count())

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 221, Finished, Available, Finished, False)

Invalid team_id_for: 0
Invalid team_id_against: 0


In [252]:
game_plays_clean_df.printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 222, Finished, Available, Finished, False)

root
 |-- play_id: string (nullable = true)
 |-- game_id: integer (nullable = true)
 |-- team_id_for: string (nullable = true)
 |-- team_id_against: string (nullable = true)
 |-- event: string (nullable = true)
 |-- secondaryType: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- period: integer (nullable = true)
 |-- periodType: string (nullable = true)
 |-- periodTime: integer (nullable = true)
 |-- periodTimeRemaining: string (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- goals_away: integer (nullable = true)
 |-- goals_home: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- st_x: string (nullable = true)
 |-- st_y: string (nullable = true)



In [253]:
from pyspark.sql.functions import col, when

game_plays_clean_df = (
    game_plays_clean_df
    .withColumn(
        "team_id_for",
        when(col("team_id_for") == "NA", None)
        .otherwise(col("team_id_for"))
        .cast("long")
    )
    .withColumn(
        "team_id_against",
        when(col("team_id_against") == "NA", None)
        .otherwise(col("team_id_against"))
        .cast("long")
    )
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 223, Finished, Available, Finished, False)

In [254]:
game_plays_clean_df.select(
    "team_id_for",
    "team_id_against"
).printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 224, Finished, Available, Finished, False)

root
 |-- team_id_for: long (nullable = true)
 |-- team_id_against: long (nullable = true)



In [255]:
invalid_team_for = validate_fk(
    game_plays_clean_df.filter(col("team_id_for").isNotNull()),
    ["team_id_for"],
    team_info_clean_df,
    ["team_id"]
)

invalid_team_against = validate_fk(
    game_plays_clean_df.filter(col("team_id_against").isNotNull()),
    ["team_id_against"],
    team_info_clean_df,
    ["team_id"]
)

print("Invalid team_id_for:", invalid_team_for.count())
print("Invalid team_id_against:", invalid_team_against.count())

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 225, Finished, Available, Finished, False)

Invalid team_id_for: 0


Invalid team_id_against: 0


The issue is NA is not being treated as a true NULL. It is a literal string value, so our FK check sees it as real team ID and flags it as orphan. We have converted "NA" in the two columns to actual NULL values. Also cast them to the same numeric type as team_info.team_id.

###### Inspect and resolve the 11 orphan play_id values

In [256]:
orphan_play_ids = (
    game_plays_players_clean_df
    .select("play_id")
    .dropDuplicates()
    .join(
        game_plays_clean_df.select("play_id").dropDuplicates(),
        on="play_id",
        how="left_anti"
    )
)

print("Number of orphan play IDs:", orphan_play_ids.count())

display(
    orphan_play_ids.orderBy("play_id")
)


StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 226, Finished, Available, Finished, False)

Number of orphan play IDs: 11


SynapseWidget(Synapse.DataFrame, d4eec134-3068-4266-85ae-fbea4ee86612)

In [257]:
orphan_play_records = (
    game_plays_players_clean_df
    .join(
        orphan_play_ids,
        on="play_id",
        how="inner"
    )
    .select(
        "play_id",
        "game_id",
        "player_id",
        "playerType"
    )
    .orderBy("game_id", "play_id", "player_id")
)

print("Total affected child records:", orphan_play_records.count())

display(orphan_play_records)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 227, Finished, Available, Finished, False)

Total affected child records: 19


SynapseWidget(Synapse.DataFrame, 9764e7a1-dc5b-4789-bc6a-886a902aea6e)

In [258]:
# Summary by game
display(
    orphan_play_records
    .groupBy("game_id")
    .count()
    .orderBy("game_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 228, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 821c88a2-9795-4d4d-9ddb-728c0c8060f4)

In [259]:
# Summary by playerType
display(
    orphan_play_records
    .groupBy("playerType")
    .count()
    .orderBy("playerType")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 229, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f574144c-0f56-4355-a205-8ec4e51418a7)

Also show the 19 records again, but grouped visually by play:

In [260]:
display(
    orphan_play_records
    .orderBy("play_id", "playerType", "player_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 230, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2ce741a-b36d-44cb-ad45-00f30c1034c0)

In [261]:
# Preserve orphan records separately for audit/data-quality purposes
game_plays_players_quarantine_df = (
    game_plays_players_clean_df
    .join(
        game_plays_clean_df.select("play_id").dropDuplicates(),
        on="play_id",
        how="left_anti"
    )
)

print(
    "Quarantined rows:",
    game_plays_players_quarantine_df.count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 231, Finished, Available, Finished, False)

Quarantined rows: 19


In [262]:
game_plays_players_clean_df = (
    game_plays_players_clean_df
    .join(
        game_plays_clean_df.select("play_id").dropDuplicates(),
        on="play_id",
        how="left_semi"
    )
)

print(
    "Silver game_plays_players rows:",
    game_plays_players_clean_df.count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 232, Finished, Available, Finished, False)

Silver game_plays_players rows: 6362785


Now we validate the FK again

In [263]:
invalid_play_ids = validate_fk(
    game_plays_players_clean_df,
    ["play_id"],
    game_plays_clean_df,
    ["play_id"]
)

print(
    "Invalid play_id after quarantine:",
    invalid_play_ids.count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 233, Finished, Available, Finished, False)

Invalid play_id after quarantine: 0


11 play_id values in game_plays_players had no corresponding parent record in game_plays, affecting 19 player-event records. These records were preserved in a quarantine dataset rather than deleted.

# **Load: Clean, Validate, convert data type and Save Silver Tables**

Step 4 - Clean, Validate and Save Silver Tables

## **silver_game**

In [264]:
from pyspark.sql.functions import col, sum, when

print("=== GAME SCHEMA ===")
game_clean_df.printSchema()

print("\n=== MISSING VALUES ===")

game_missing = game_clean_df.select([
    sum(
        when(
            col(c).isNull() |
            (col(c).cast("string") == "") |
            (col(c).cast("string") == "NA"),
            1
        ).otherwise(0)
    ).alias(c)
    for c in game_clean_df.columns
])

display(game_missing)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 234, Finished, Available, Finished, False)

=== GAME SCHEMA ===
root
 |-- game_id: integer (nullable = true)
 |-- season: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- date_time_GMT: timestamp (nullable = true)
 |-- away_team_id: integer (nullable = true)
 |-- home_team_id: integer (nullable = true)
 |-- away_goals: integer (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- outcome: string (nullable = true)
 |-- home_rink_side_start: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- venue_link: string (nullable = true)
 |-- venue_time_zone_id: string (nullable = true)
 |-- venue_time_zone_offset: integer (nullable = true)
 |-- venue_time_zone_tz: string (nullable = true)


=== MISSING VALUES ===


SynapseWidget(Synapse.DataFrame, b6bf1c1c-ef25-45c6-8c30-8705e7ba5406)

In [265]:
print(
    "Negative away_goals:",
    game_clean_df.filter(col("away_goals") < 0).count()
)

print(
    "Negative home_goals:",
    game_clean_df.filter(col("home_goals") < 0).count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 235, Finished, Available, Finished, False)

Negative away_goals: 0
Negative home_goals: 0


In [266]:
game_clean_df.select(
    "date_time_GMT"
).printSchema()

game_clean_df.select(
    "date_time_GMT"
).show(10, truncate=False)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 236, Finished, Available, Finished, False)

root
 |-- date_time_GMT: timestamp (nullable = true)

+-------------------+
|date_time_GMT      |
+-------------------+
|2018-03-02 02:00:00|
|2018-01-21 01:30:00|
|2015-10-31 23:00:00|
|2017-11-16 03:00:00|
|2016-03-18 00:30:00|
|2018-04-07 23:00:00|
|2017-11-26 19:00:00|
|2017-10-13 02:00:00|
|2016-10-22 23:00:00|
|2016-10-25 23:00:00|
+-------------------+
only showing top 10 rows



###### Venue TimeZone Checks

Profile venue - timezone combinations across the entire game dataframe

In [ ]:
from pyspark.sql.functions import col, count

venue_timezone_check = (
    game_clean_df
    .groupBy(
        "venue",
        "venue_time_zone_id"
    )
    .count()
    .orderBy(
        "venue",
        col("count").desc()
    )
)

display(venue_timezone_check)

Venues associated with more than one timezone ID:

In [ ]:
from pyspark.sql.functions import countDistinct

multi_timezone_venues = (
    game_clean_df
    .groupBy("venue")
    .agg(
        countDistinct("venue_time_zone_id")
        .alias("timezone_count")
    )
    .filter(col("timezone_count") > 1)
    .orderBy(col("timezone_count").desc())
)

display(multi_timezone_venues)

Identify the Venues having timezones problem

In [ ]:
problem_venue_timezones = (
    game_clean_df
    .join(
        multi_timezone_venues.select("venue"),
        on="venue",
        how="inner"
    )
    .groupBy(
        "venue",
        "venue_time_zone_id"
    )
    .count()
    .orderBy(
        "venue",
        col("count").desc()
    )
)

display(problem_venue_timezones)

Rogers Place Time Zones

In [ ]:
display(
    game_clean_df
    .filter(col("venue") == "Rogers Place")
    .groupBy(
        "venue_time_zone_id",
        "venue_time_zone_offset",
        "venue_time_zone_tz"
    )
    .count()
    .orderBy("venue_time_zone_id", "venue_time_zone_offset")
)

Find whether Rogers Place appears in seasons before 2016-2017?

In [ ]:
from pyspark.sql.functions import min, max

game_clean_df \
    .filter(col("venue") == "Rogers Place") \
    .select(
        min("date_time_GMT").alias("earliest_game"),
        max("date_time_GMT").alias("latest_game")
    ) \
    .show(truncate=False)

Check whether the incorrect timezone is actually associated with the away team/location?

In [ ]:
from pyspark.sql.functions import col

display(
    game_clean_df
    .filter(
        (col("venue") == "Rogers Place") &
        (col("venue_time_zone_id") != "America/Edmonton")
    )
    .select(
        "game_id",
        "date_time_GMT",
        "away_team_id",
        "home_team_id",
        "venue",
        "venue_time_zone_id",
        "venue_time_zone_offset",
        "venue_time_zone_tz"
    )
    .orderBy("date_time_GMT")

The 73 non-Edmonton Rogers Place records are all from August–September 2020. These are from the NHL's 2020 COVID bubble/playoff period, when Edmonton's Rogers Place was used as a hub venue for teams that normally played in many different cities.

In [ ]:
from pyspark.sql.functions import col, countDistinct

multi_timezone_venues_after = (
    game_clean_df
    .groupBy("venue")
    .agg(
        countDistinct("venue_time_zone_id")
        .alias("timezone_id_count")
    )
    .filter(col("timezone_id_count") > 1)
    .orderBy(col("timezone_id_count").desc())
)

display(multi_timezone_venues_after)

In [ ]:
problem_venue_details = (
    game_clean_df
    .join(
        multi_timezone_venues_after.select("venue"),
        on="venue",
        how="inner"
    )
    .groupBy(
        "venue",
        "venue_time_zone_id",
        "venue_time_zone_offset",
        "venue_time_zone_tz"
    )
    .count()
    .orderBy(
        "venue",
        "venue_time_zone_id",
        "venue_time_zone_offset"
    )
)

display(problem_venue_details)

In [ ]:
display(
    game_clean_df
    .join(
        multi_timezone_venues_after.select("venue"),
        on="venue",
        how="inner"
    )
    .groupBy(
        "venue",
        "venue_time_zone_id"
    )
    .agg(
        min("date_time_GMT").alias("earliest_game"),
        max("date_time_GMT").alias("latest_game"),
        count("*").alias("rows")
    )
    .orderBy("venue", "venue_time_zone_id")
)

In [ ]:
display(
    game_clean_df
    .filter(
        (col("venue").startswith("Enterprise")) &
        (col("venue_time_zone_id") == "America/New_York")
    )
    .select(
        "game_id",
        "date_time_GMT",
        "away_team_id",
        "home_team_id",
        "venue",
        "venue_time_zone_id",
        "venue_time_zone_offset",
        "venue_time_zone_tz"
    )
)

In [ ]:
display(
    game_clean_df
    .filter(
        (col("venue").startswith("SAP Center")) &
        (col("venue_time_zone_id") == "America/New_York")
    )
    .select(
        "game_id",
        "date_time_GMT",
        "away_team_id",
        "home_team_id",
        "venue",
        "venue_time_zone_id",
        "venue_time_zone_offset",
        "venue_time_zone_tz"
    )
)

Ran checks on Enterprise Center and SAP Center to see a pattern. The records turned our to be on NHL All-Star games/events. No normal games involving the regular home team of Enterprise Center or SAP Center.

###### Final Checks

In [267]:
print(
    "NULL game_id:",
    game_clean_df
        .filter(col("game_id").isNull())
        .count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 237, Finished, Available, Finished, False)

NULL game_id: 0


In [268]:
# Final validation: 

total_rows = game_clean_df.count()
unique_game_ids = game_clean_df.select("game_id").distinct().count()
null_game_ids = game_clean_df.filter(col("game_id").isNull()).count()

negative_away_goals = game_clean_df.filter(col("away_goals") < 0).count()
negative_home_goals = game_clean_df.filter(col("home_goals") < 0).count()

null_dates = game_clean_df.filter(col("date_time_GMT").isNull()).count()

print("Total rows:", total_rows)
print("Unique game_id:", unique_game_ids)
print("NULL game_id:", null_game_ids)
print("NULL date_time_GMT:", null_dates)
print("Negative away_goals:", negative_away_goals)
print("Negative home_goals:", negative_home_goals)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 238, Finished, Available, Finished, False)

Total rows: 23735
Unique game_id: 23735
NULL game_id: 0
NULL date_time_GMT: 0
Negative away_goals: 0
Negative home_goals: 0


In [269]:
game_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_game")

print("silver_game saved successfully.")

StatementMeta(, 5a5641b6-3a20-4785-b67e-b70a7ccba1a5, 322, Finished, Available, Finished, False)

silver_game saved successfully.


In [270]:
silver_game_df = spark.table("silver_game")

print("Saved rows:", silver_game_df.count())
print("Saved columns:", len(silver_game_df.columns))

silver_game_df.printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 239, Finished, Available, Finished, False)

Saved rows: 23735
Saved columns: 15
root
 |-- game_id: integer (nullable = true)
 |-- season: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- date_time_GMT: timestamp (nullable = true)
 |-- away_team_id: integer (nullable = true)
 |-- home_team_id: integer (nullable = true)
 |-- away_goals: integer (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- outcome: string (nullable = true)
 |-- home_rink_side_start: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- venue_link: string (nullable = true)
 |-- venue_time_zone_id: string (nullable = true)
 |-- venue_time_zone_offset: integer (nullable = true)
 |-- venue_time_zone_tz: string (nullable = true)



## **silver_team_info** (Freezed, using silver table from dataflow gen2)

In [118]:

print("=== TEAM_INFO SCHEMA ===")
team_info_clean_df.printSchema()

print("\n=== TEAM_INFO ROW COUNT ===")
print("Rows:", team_info_clean_df.count())

print("\n=== TEAM_INFO MISSING VALUES ===")

team_info_missing = team_info_clean_df.select([
    sum(
        when(
            col(c).isNull() |
            (col(c).cast("string") == "") |
            (col(c).cast("string") == "NA"),
            1
        ).otherwise(0)
    ).alias(c)
    for c in team_info_clean_df.columns
])

display(team_info_missing)

StatementMeta(, b887efee-a773-491e-be3c-920dc13086fb, 120, Finished, Available, Finished, False)

=== TEAM_INFO SCHEMA ===
root
 |-- team_id: integer (nullable = true)
 |-- franchiseId: integer (nullable = true)
 |-- shortName: string (nullable = true)
 |-- teamName: string (nullable = true)
 |-- abbreviation: string (nullable = true)
 |-- link: string (nullable = true)


=== TEAM_INFO ROW COUNT ===
Rows: 37

=== TEAM_INFO MISSING VALUES ===


SynapseWidget(Synapse.DataFrame, 671b4378-4f08-456d-9fa1-49d61a57fc7c)

In [119]:
# team_info Primary Key Validation

total_rows = team_info_clean_df.count()
unique_team_ids = team_info_clean_df.select("team_id").distinct().count()
null_team_ids = team_info_clean_df.filter(col("team_id").isNull()).count()

print("Total rows:", total_rows)
print("Unique team_id:", unique_team_ids)
print("NULL team_id:", null_team_ids)

StatementMeta(, b887efee-a773-491e-be3c-920dc13086fb, 121, Finished, Available, Finished, False)

Total rows: 37
Unique team_id: 37
NULL team_id: 0


In [120]:
# info of the added 4 teams
display(
    team_info_clean_df
    .filter(col("team_id").isin([87, 88, 89, 90]))
    .orderBy("team_id")
)

StatementMeta(, b887efee-a773-491e-be3c-920dc13086fb, 122, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 544c1d86-aa3a-421c-aea0-1e1294896c50)

In [121]:
team_info_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_team_info")

print("silver_team_info saved successfully.")

StatementMeta(, b887efee-a773-491e-be3c-920dc13086fb, 123, Finished, Available, Finished, False)

silver_team_info saved successfully.


In [122]:
silver_team_info_df = spark.table("silver_team_info")

print("Saved rows:", silver_team_info_df.count())
print("Saved columns:", len(silver_team_info_df.columns))

silver_team_info_df.printSchema()

StatementMeta(, b887efee-a773-491e-be3c-920dc13086fb, 124, Finished, Available, Finished, False)

Saved rows: 37
Saved columns: 6
root
 |-- team_id: integer (nullable = true)
 |-- franchiseId: integer (nullable = true)
 |-- shortName: string (nullable = true)
 |-- teamName: string (nullable = true)
 |-- abbreviation: string (nullable = true)
 |-- link: string (nullable = true)



## **silver_player_info**

In [271]:

print("=== PLAYER_INFO SCHEMA ===")
player_info_clean_df.printSchema()

print("\n=== PLAYER_INFO ROW COUNT ===")
print("Rows:", player_info_clean_df.count())

print("\n=== PLAYER_INFO MISSING VALUES ===")

player_info_missing = player_info_clean_df.select([
    sum(
        when(
            col(c).isNull() |
            (col(c).cast("string") == "") |
            (col(c).cast("string") == "NA"),
            1
        ).otherwise(0)
    ).alias(c)
    for c in player_info_clean_df.columns
])

display(player_info_missing)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 240, Finished, Available, Finished, False)

=== PLAYER_INFO SCHEMA ===
root
 |-- player_id: integer (nullable = true)
 |-- firstName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- birthCity: string (nullable = true)
 |-- primaryPosition: string (nullable = true)
 |-- birthDate: timestamp (nullable = true)
 |-- birthStateProvince: string (nullable = true)
 |-- height: string (nullable = true)
 |-- height_cm: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- shootsCatches: string (nullable = true)


=== PLAYER_INFO ROW COUNT ===
Rows: 3925

=== PLAYER_INFO MISSING VALUES ===


SynapseWidget(Synapse.DataFrame, 7cbaa2bb-9914-4f86-a426-4eac0214cf7f)

In [272]:

total_rows = player_info_clean_df.count()
unique_player_ids = player_info_clean_df.select("player_id").distinct().count()
null_player_ids = player_info_clean_df.filter(col("player_id").isNull()).count()

negative_height = player_info_clean_df.filter(col("height_cm") < 0).count()
negative_weight = player_info_clean_df.filter(col("weight") < 0).count()

print("Total rows:", total_rows)
print("Unique player_id:", unique_player_ids)
print("NULL player_id:", null_player_ids)
print("Negative height_cm:", negative_height)
print("Negative weight:", negative_weight)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 241, Finished, Available, Finished, False)

Total rows: 3925
Unique player_id: 3925
NULL player_id: 0
Negative height_cm: 0
Negative weight: 0


In [273]:
player_info_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_player_info")

print("silver_player_info saved successfully.")

StatementMeta(, 5a5641b6-3a20-4785-b67e-b70a7ccba1a5, 326, Finished, Available, Finished, False)

silver_player_info saved successfully.


In [274]:
silver_player_info_df = spark.table("silver_player_info")

print("Saved rows:", silver_player_info_df.count())
print("Saved columns:", len(silver_player_info_df.columns))

silver_player_info_df.printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 242, Finished, Available, Finished, False)

Saved rows: 3925
Saved columns: 12
root
 |-- player_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- birth_city: string (nullable = true)
 |-- primary_position: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- birth_state_province: string (nullable = true)
 |-- height: string (nullable = true)
 |-- height_cm: double (nullable = true)
 |-- weight: long (nullable = true)
 |-- shoots_catches: string (nullable = true)



## **silver_game_teams_stats**

### **Tanvir**

In [275]:
# ============================================================
# SILVER TABLE 4: GAME_TEAMS_STATS
# ============================================================

print("=== GAME_TEAMS_STATS SCHEMA ===")
game_teams_stats_clean_df.printSchema()

print("\n=== ROW COUNT ===")
print("Rows:", game_teams_stats_clean_df.count())

print("\n=== MISSING VALUES ===")

game_teams_stats_missing = game_teams_stats_clean_df.select([
    sum(
        when(
            col(c).isNull() |
            (col(c).cast("string") == "") |
            (col(c).cast("string") == "NA"),
            1
        ).otherwise(0)
    ).alias(c)
    for c in game_teams_stats_clean_df.columns
])

display(game_teams_stats_missing)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 243, Finished, Available, Finished, False)

=== GAME_TEAMS_STATS SCHEMA ===
root
 |-- game_id: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- HoA: string (nullable = true)
 |-- won: boolean (nullable = true)
 |-- settled_in: string (nullable = true)
 |-- head_coach: string (nullable = true)
 |-- goals: string (nullable = true)
 |-- shots: string (nullable = true)
 |-- hits: string (nullable = true)
 |-- pim: string (nullable = true)
 |-- powerPlayOpportunities: string (nullable = true)
 |-- powerPlayGoals: string (nullable = true)
 |-- faceOffWinPercentage: string (nullable = true)
 |-- giveaways: string (nullable = true)
 |-- takeaways: string (nullable = true)
 |-- blocked: string (nullable = true)
 |-- startRinkSide: string (nullable = true)


=== ROW COUNT ===
Rows: 47470

=== MISSING VALUES ===


SynapseWidget(Synapse.DataFrame, 04a416a5-1b26-42f4-a096-c03a5a74720d)

In [276]:
from pyspark.sql.functions import col

missing_core_stats = (
    game_teams_stats_clean_df
    .filter(
        col("goals").isNull() |
        col("shots").isNull() |
        col("pim").isNull() |
        col("powerPlayOpportunities").isNull() |
        col("powerPlayGoals").isNull()
    )
)

print("Affected rows:", missing_core_stats.count())

display(
    missing_core_stats
    .orderBy("game_id", "team_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 244, Finished, Available, Finished, False)

Affected rows: 0


SynapseWidget(Synapse.DataFrame, 1c2a5322-066f-4144-a11c-036dcff6174b)

In [277]:
missing_core_with_game = (
    missing_core_stats.alias("s")
    .join(
        game_clean_df
        .select("game_id", "season", "type", "date_time_GMT")
        .alias("g"),
        on="game_id",
        how="left"
    )
)

display(
    missing_core_with_game
    .select(
        "game_id",
        "season",
        "type",
        "date_time_GMT",
        "team_id",
        "HoA",
        "won",
        "goals",
        "shots",
        "pim",
        "powerPlayOpportunities",
        "powerPlayGoals"
    )
    .orderBy("game_id", "team_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 245, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f02b855b-1b38-4247-a41d-a9f56f09b98f)

In [278]:
game_teams_stats_clean_df.select(
    "goals",
    "shots",
    "pim",
    "powerPlayOpportunities",
    "powerPlayGoals",
    "hits",
    "faceOffWinPercentage",
    "giveaways",
    "takeaways",
    "blocked"
).printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 246, Finished, Available, Finished, False)

root
 |-- goals: string (nullable = true)
 |-- shots: string (nullable = true)
 |-- pim: string (nullable = true)
 |-- powerPlayOpportunities: string (nullable = true)
 |-- powerPlayGoals: string (nullable = true)
 |-- hits: string (nullable = true)
 |-- faceOffWinPercentage: string (nullable = true)
 |-- giveaways: string (nullable = true)
 |-- takeaways: string (nullable = true)
 |-- blocked: string (nullable = true)



In [279]:
from pyspark.sql.functions import col

na_core_stats = (
    game_teams_stats_clean_df
    .filter(
        (col("goals").cast("string") == "NA") |
        (col("shots").cast("string") == "NA") |
        (col("pim").cast("string") == "NA") |
        (col("powerPlayOpportunities").cast("string") == "NA") |
        (col("powerPlayGoals").cast("string") == "NA")
    )
)

print("Rows containing 'NA':", na_core_stats.count())

display(
    na_core_stats
    .select(
        "game_id",
        "team_id",
        "HoA",
        "goals",
        "shots",
        "pim",
        "powerPlayOpportunities",
        "powerPlayGoals"
    )
    .orderBy("game_id", "team_id")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 247, Finished, Available, Finished, False)

Rows containing 'NA': 8


SynapseWidget(Synapse.DataFrame, ff9c7989-1949-4050-9c03-1b6c42ba8f72)

In [280]:
game_teams_stats_clean_df.select(
    "goals",
    "shots",
    "pim",
    "powerPlayOpportunities",
    "powerPlayGoals"
).printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 248, Finished, Available, Finished, False)

root
 |-- goals: string (nullable = true)
 |-- shots: string (nullable = true)
 |-- pim: string (nullable = true)
 |-- powerPlayOpportunities: string (nullable = true)
 |-- powerPlayGoals: string (nullable = true)



In [281]:
from pyspark.sql.functions import col, when

core_numeric_columns = [
    "goals",
    "shots",
    "pim",
    "powerPlayOpportunities",
    "powerPlayGoals"
]

for c in core_numeric_columns:
    game_teams_stats_clean_df = (
        game_teams_stats_clean_df
        .withColumn(
            c,
            when(
                col(c).isin("NA", ""),
                None
            )
            .otherwise(col(c))
            .cast("integer")
        )
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 249, Finished, Available, Finished, False)

In [282]:
game_teams_stats_clean_df.select(
    "goals",
    "shots",
    "pim",
    "powerPlayOpportunities",
    "powerPlayGoals"
).printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 250, Finished, Available, Finished, False)

root
 |-- goals: integer (nullable = true)
 |-- shots: integer (nullable = true)
 |-- pim: integer (nullable = true)
 |-- powerPlayOpportunities: integer (nullable = true)
 |-- powerPlayGoals: integer (nullable = true)



In [283]:
for c in core_numeric_columns:
    print(
        f"{c} NULL:",
        game_teams_stats_clean_df
        .filter(col(c).isNull())
        .count()
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 251, Finished, Available, Finished, False)

goals NULL: 8
shots NULL: 8
pim NULL: 8
powerPlayOpportunities NULL: 8
powerPlayGoals NULL: 8


In [284]:
# Check actual NULLs

columns_4928 = [
    "hits",
    "giveaways",
    "takeaways",
    "blocked"
]

for c in columns_4928:
    print(
        f"{c} NULL:",
        game_teams_stats_clean_df
        .filter(col(c).isNull())
        .count()
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 252, Finished, Available, Finished, False)

hits NULL: 0
giveaways NULL: 0
takeaways NULL: 0
blocked NULL: 0


In [285]:
for c in columns_4928:
    print(
        f"{c} NA:",
        game_teams_stats_clean_df
        .filter(col(c).cast("string") == "NA")
        .count()
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 253, Finished, Available, Finished, False)

hits NA: 4928
giveaways NA: 4928
takeaways NA: 4928
blocked NA: 4928


In [286]:
missing_four_stats = (
    game_teams_stats_clean_df
    .filter(
        (col("hits").cast("string") == "NA") &
        (col("giveaways").cast("string") == "NA") &
        (col("takeaways").cast("string") == "NA") &
        (col("blocked").cast("string") == "NA")
    )
)

print(
    "Rows where all four statistics are NA:",
    missing_four_stats.count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 254, Finished, Available, Finished, False)

Rows where all four statistics are NA: 4928


**Adding a copy of my silver_game Delta table to another Lakehouse.**

In [287]:
%%sql
SHOW DATABASES

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 255, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [288]:
missing_four_with_game = (
    missing_four_stats.alias("s")
    .join(
        game_clean_df
        .select("game_id", "season", "type", "date_time_GMT")
        .alias("g"),
        on="game_id",
        how="left"
    )
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 256, Finished, Available, Finished, False)

In [289]:
display(
    missing_four_with_game
    .groupBy("season")
    .count()
    .orderBy("season")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 257, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09e9f075-ead6-4543-9ff1-b8b4ce56456d)

In [290]:
display(
    missing_four_with_game
    .groupBy("season", "type")
    .count()
    .orderBy("season", "type")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 258, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 82db385b-7aa9-421d-827c-374a3fabf0b4)

In [291]:
from pyspark.sql.functions import min, max

missing_four_with_game.select(
    min("date_time_GMT").alias("earliest_date"),
    max("date_time_GMT").alias("latest_date")
).show()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 259, Finished, Available, Finished, False)

+-------------------+-------------------+
|      earliest_date|        latest_date|
+-------------------+-------------------+
|2000-10-04 23:00:00|2018-05-25 00:00:00|
+-------------------+-------------------+



In [292]:
from pyspark.sql.functions import col, when

stats_columns = [
    "hits",
    "giveaways",
    "takeaways",
    "blocked"
]

for c in stats_columns:
    game_teams_stats_clean_df = (
        game_teams_stats_clean_df
        .withColumn(
            c,
            when(
                col(c).isin("NA", ""),
                None
            )
            .otherwise(col(c))
            .cast("integer")
        )
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 260, Finished, Available, Finished, False)

In [293]:
for c in stats_columns:
    print(
        f"{c} NULL:",
        game_teams_stats_clean_df
        .filter(col(c).isNull())
        .count()
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 261, Finished, Available, Finished, False)

hits NULL: 4928
giveaways NULL: 4928
takeaways NULL: 4928
blocked NULL: 4928


In [294]:
game_teams_stats_clean_df.select(
    "hits",
    "giveaways",
    "takeaways",
    "blocked"
).printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 262, Finished, Available, Finished, False)

root
 |-- hits: integer (nullable = true)
 |-- giveaways: integer (nullable = true)
 |-- takeaways: integer (nullable = true)
 |-- blocked: integer (nullable = true)



In [295]:
print(
    "faceOffWinPercentage NULL:",
    game_teams_stats_clean_df
    .filter(col("faceOffWinPercentage").isNull())
    .count()
)

print(
    "faceOffWinPercentage NA:",
    game_teams_stats_clean_df
    .filter(col("faceOffWinPercentage").cast("string") == "NA")
    .count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 263, Finished, Available, Finished, False)

faceOffWinPercentage NULL: 0
faceOffWinPercentage NA: 22148


In [296]:
faceoff_missing = (
    game_teams_stats_clean_df
    .filter(
        (col("faceOffWinPercentage").cast("string") == "NA") |
        col("faceOffWinPercentage").isNull()
    )
)

faceoff_missing_with_game = (
    faceoff_missing.alias("s")
    .join(
        game_clean_df
        .select("game_id", "season", "type", "date_time_GMT")
        .alias("g"),
        on="game_id",
        how="left"
    )
)

display(
    faceoff_missing_with_game
    .groupBy("season")
    .count()
    .orderBy("season")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 264, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ea207fc-4035-49bb-9fd6-1e7323332a0b)

In [297]:
display(
    faceoff_missing_with_game
    .groupBy("season", "type")
    .count()
    .orderBy("season", "type")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 265, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dccd7b3a-0740-48a4-a6f8-95e61ad0c060)

In [298]:
from pyspark.sql.functions import min, max

faceoff_missing_with_game.select(
    min("date_time_GMT").alias("earliest_date"),
    max("date_time_GMT").alias("latest_date")
).show()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 266, Finished, Available, Finished, False)

+-------------------+-------------------+
|      earliest_date|        latest_date|
+-------------------+-------------------+
|2000-10-04 23:00:00|2018-05-25 00:00:00|
+-------------------+-------------------+



In [299]:
from pyspark.sql.functions import col, when

game_teams_stats_clean_df = (
    game_teams_stats_clean_df
    .withColumn(
        "faceOffWinPercentage",
        when(
            col("faceOffWinPercentage").isin("NA", ""),
            None
        )
        .otherwise(col("faceOffWinPercentage"))
        .cast("double")
    )
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 267, Finished, Available, Finished, False)

In [300]:
print(
    "faceOffWinPercentage NULL:",
    game_teams_stats_clean_df
    .filter(col("faceOffWinPercentage").isNull())
    .count()
)

game_teams_stats_clean_df.select(
    "faceOffWinPercentage"
).printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 268, Finished, Available, Finished, False)

faceOffWinPercentage NULL: 22148
root
 |-- faceOffWinPercentage: double (nullable = true)



In [301]:
from pyspark.sql.functions import col

for c in ["head_coach", "startRinkSide"]:

    null_count = (
        game_teams_stats_clean_df
        .filter(col(c).isNull())
        .count()
    )

    na_count = (
        game_teams_stats_clean_df
        .filter(col(c).cast("string") == "NA")
        .count()
    )

    empty_count = (
        game_teams_stats_clean_df
        .filter(col(c).cast("string") == "")
        .count()
    )

    print(
        f"{c}: NULL={null_count}, NA={na_count}, Empty={empty_count}"
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 269, Finished, Available, Finished, False)

head_coach: NULL=0, NA=28, Empty=0
startRinkSide: NULL=0, NA=2198, Empty=0


In [302]:
descriptive_missing = (
    game_teams_stats_clean_df
    .filter(
        col("head_coach").isNull() |
        (col("head_coach") == "NA") |
        (col("head_coach") == "") |
        col("startRinkSide").isNull() |
        (col("startRinkSide") == "NA") |
        (col("startRinkSide") == "")
    )
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 270, Finished, Available, Finished, False)

In [303]:
descriptive_missing_with_game = (
    descriptive_missing.alias("s")
    .join(
        game_clean_df
        .select(
            "game_id",
            "season",
            "type",
            "date_time_GMT"
        )
        .alias("g"),
        on="game_id",
        how="left"
    )
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 271, Finished, Available, Finished, False)

In [304]:
display(
    descriptive_missing_with_game
    .groupBy("season", "type")
    .count()
    .orderBy("season", "type")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 272, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3dee8dae-4612-4158-971f-82622459c001)

In [305]:
for c in ["head_coach", "startRinkSide"]:

    missing_df = (
        game_teams_stats_clean_df
        .filter(
            col(c).isNull() |
            (col(c) == "NA") |
            (col(c) == "")
        )
        .join(
            game_clean_df.select(
                "game_id",
                "season",
                "type",
                "date_time_GMT"
            ),
            on="game_id",
            how="left"
        )
    )

    print(f"\nMissing pattern for: {c}")

    missing_df.groupBy(
        "season", "type"
    ).count().orderBy(
        "season", "type"
    ).show(100, truncate=False)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 273, Finished, Available, Finished, False)


Missing pattern for: head_coach
+--------+----+-----+
|season  |type|count|
+--------+----+-----+
|20162017|P   |12   |
|20172018|P   |16   |
+--------+----+-----+


Missing pattern for: startRinkSide
+--------+----+-----+
|season  |type|count|
+--------+----+-----+
|20002001|R   |164  |
|20012002|R   |164  |
|20022003|R   |164  |
|20032004|R   |164  |
|20052006|R   |164  |
|20062007|R   |164  |
|20072008|R   |164  |
|20082009|R   |164  |
|20092010|R   |164  |
|20102011|P   |4    |
|20102011|R   |164  |
|20112012|P   |18   |
|20112012|R   |82   |
|20122013|R   |48   |
|20132014|R   |82   |
|20162017|P   |12   |
|20172018|P   |36   |
|20172018|R   |82   |
|20182019|A   |4    |
|20182019|P   |6    |
|20182019|R   |82   |
|20192020|A   |6    |
|20192020|P   |20   |
|20192020|R   |76   |
+--------+----+-----+



In [306]:
from pyspark.sql.functions import col, when

descriptive_columns = [
    "head_coach",
    "startRinkSide"
]

for c in descriptive_columns:
    game_teams_stats_clean_df = (
        game_teams_stats_clean_df
        .withColumn(
            c,
            when(
                col(c).isin("NA", ""),
                None
            )
            .otherwise(col(c))
            .cast("string")
        )
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 274, Finished, Available, Finished, False)

In [307]:
for c in descriptive_columns:
    print(
        f"{c} NULL:",
        game_teams_stats_clean_df
        .filter(col(c).isNull())
        .count()
    )

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 275, Finished, Available, Finished, False)

head_coach NULL: 28
startRinkSide NULL: 2198


In [308]:
game_teams_stats_clean_df.select(
    "head_coach",
    "startRinkSide"
).printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 276, Finished, Available, Finished, False)

root
 |-- head_coach: string (nullable = true)
 |-- startRinkSide: string (nullable = true)



In [314]:
# Elias
# Visual check for columns with nulls and NA.

columns_with_nulls = ["head_coach", "goals", "shots", "hits", "pim", "powerPlayOpportunities", "powerPlayGoals", "faceOffWinPercentage", "giveaways", "takeaways", "blocked", "startRinkSide"]

for column in columns_with_nulls:
    null = game_teams_stats_clean_df.filter(col(column).isNull())
    null_count = null.count()
    null.show(10)
    print(f"Above dataframe is {column}: {null_count} Nulls")

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 277, Finished, Available, Finished, False)

+----------+-------+----+-----+----------+----------+-----+-----+----+----+----------------------+--------------+--------------------+---------+---------+-------+-------------+
|   game_id|team_id| HoA|  won|settled_in|head_coach|goals|shots|hits| pim|powerPlayOpportunities|powerPlayGoals|faceOffWinPercentage|giveaways|takeaways|blocked|startRinkSide|
+----------+-------+----+-----+----------+----------+-----+-----+----+----+----------------------+--------------+--------------------+---------+---------+-------+-------------+
|2017030416|     15|home|false|       tbc|      NULL|    0|    0|   0|   0|                     0|             0|                 0.0|        0|        0|      0|         NULL|
|2016030137|     10|away|false|       tbc|      NULL|    0|    0|   0|   0|                     0|             0|                 0.0|        0|        0|      0|         NULL|
|2016030417|      5|home|false|       tbc|      NULL|    0|    0|   0|   0|                     0|             0|  

+----------+-------+----+-----+----------+---------------+-----+-----+----+---+----------------------+--------------+--------------------+---------+---------+-------+-------------+
|   game_id|team_id| HoA|  won|settled_in|     head_coach|goals|shots|hits|pim|powerPlayOpportunities|powerPlayGoals|faceOffWinPercentage|giveaways|takeaways|blocked|startRinkSide|
+----------+-------+----+-----+----------+---------------+-----+-----+----+---+----------------------+--------------+--------------------+---------+---------+-------+-------------+
|2000020460|      5|home|false|       REG|    Ivan Hlinka|    1|   36|NULL| 29|                     6|             0|                NULL|     NULL|     NULL|   NULL|        right|
|2000020651|      8|home| true|       REG|Michel Therrien|    5|   30|NULL| 12|                     9|             3|                NULL|     NULL|     NULL|   NULL|         left|
|2000021004|      8|away|false|       REG|Michel Therrien|    2|   29|NULL| 14|                

In [315]:
# Elias
# Closer look at head_coach, all rows with NA
game_teams_stats_clean_df.filter(col("head_coach").isNull()).sort("game_id").show(30)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 278, Finished, Available, Finished, False)

+----------+-------+----+-----+----------+----------+-----+-----+----+----+----------------------+--------------+--------------------+---------+---------+-------+-------------+
|   game_id|team_id| HoA|  won|settled_in|head_coach|goals|shots|hits| pim|powerPlayOpportunities|powerPlayGoals|faceOffWinPercentage|giveaways|takeaways|blocked|startRinkSide|
+----------+-------+----+-----+----------+----------+-----+-----+----+----+----------------------+--------------+--------------------+---------+---------+-------+-------------+
|2016030137|     10|away|false|       tbc|      NULL|    0|    0|   0|   0|                     0|             0|                 0.0|        0|        0|      0|         NULL|
|2016030137|     15|home|false|       tbc|      NULL|    0|    0|   0|   0|                     0|             0|                 0.0|        0|        0|      0|         NULL|
|2016030155|     18|away|false|       tbc|      NULL|    0|    0|   0|   0|                     0|             0|  

In [319]:
# Elias
# The 8 records with nulls across goals, shots, hits, penalty_in_minutes, power_play_opportunities, power_play_goals, face_off_win_percentage, giveaways, takeaways, blocked are the same records
# Since most of the column values are null for the 8 records and they are insignificant compared to the unique records of 47470, will drop the 8 records with nulls
gts_dropna_goals = game_teams_stats_clean_df.dropna(subset="goals")
assert gts_dropna_goals.count() == game_teams_stats_clean_df.count() - game_teams_stats_clean_df.filter((col("goals").isNull())).count()
gts_dropna_goals.count()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 279, Finished, Available, Finished, False)

47462

In [320]:
# Elias
# Drop all rows with NA for head_coach because the values across the fields for home/away records does not represent anything.
gts_dropna_head_coach = gts_dropna_goals.dropna(subset="head_coach")
assert gts_dropna_head_coach.count() == gts_dropna_goals.count() - gts_dropna_goals.filter(col("head_coach").isNull()).count()
gts_dropna_head_coach.count()

# For the other records with nulls/NA, will retain them since there are still values for important columns like goals and shots for analysis

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 280, Finished, Available, Finished, False)

47442

In [321]:
game_teams_stats_clean_df = gts_dropna_head_coach

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 281, Finished, Available, Finished, False)

### **Final Validation**

In [322]:
from pyspark.sql.functions import col, count, countDistinct

total_rows = game_teams_stats_clean_df.count()

unique_pk = (
    game_teams_stats_clean_df
    .select("game_id", "team_id")
    .distinct()
    .count()
)

null_game_id = (
    game_teams_stats_clean_df
    .filter(col("game_id").isNull())
    .count()
)

null_team_id = (
    game_teams_stats_clean_df
    .filter(col("team_id").isNull())
    .count()
)

print("Total rows:", total_rows)
print("Unique (game_id, team_id):", unique_pk)
print("NULL game_id:", null_game_id)
print("NULL team_id:", null_team_id)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 282, Finished, Available, Finished, False)

Total rows: 47442
Unique (game_id, team_id): 47442
NULL game_id: 0
NULL team_id: 0


In [323]:
invalid_game_fk = (
    game_teams_stats_clean_df
    .select("game_id")
    .dropDuplicates()
    .join(
        game_clean_df.select("game_id").dropDuplicates(),
        on="game_id",
        how="left_anti"
    )
)

invalid_team_fk = (
    game_teams_stats_clean_df
    .select("team_id")
    .dropDuplicates()
    .join(
        team_info_clean_df.select("team_id").dropDuplicates(),
        on="team_id",
        how="left_anti"
    )
)

print("Invalid game_id FK:", invalid_game_fk.count())
print("Invalid team_id FK:", invalid_team_fk.count())

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 283, Finished, Available, Finished, False)

Invalid game_id FK: 0
Invalid team_id FK: 0


In [324]:
from pyspark.sql.functions import sum, when

na_check = game_teams_stats_clean_df.select([
    sum(
        when(
            col(c).cast("string") == "NA",
            1
        ).otherwise(0)
    ).alias(c)
    for c in game_teams_stats_clean_df.columns
])

display(na_check)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 284, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bbcfbc2c-133e-49e2-ac00-73fc0cb64542)

In [325]:
numeric_count_columns = [
    "goals",
    "shots",
    "hits",
    "pim",
    "powerPlayOpportunities",
    "powerPlayGoals",
    "giveaways",
    "takeaways",
    "blocked"
]

for c in numeric_count_columns:
    negative_count = (
        game_teams_stats_clean_df
        .filter(col(c) < 0)
        .count()
    )

    print(f"{c} negative values:", negative_count)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 285, Finished, Available, Finished, False)

goals negative values: 0
shots negative values: 0
hits negative values: 0
pim negative values: 0
powerPlayOpportunities negative values: 0
powerPlayGoals negative values: 0
giveaways negative values: 0
takeaways negative values: 0
blocked negative values: 0


In [326]:
invalid_faceoff = (
    game_teams_stats_clean_df
    .filter(
        (col("faceOffWinPercentage") < 0) |
        (col("faceOffWinPercentage") > 100)
    )
)

print(
    "Invalid faceOffWinPercentage:",
    invalid_faceoff.count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 286, Finished, Available, Finished, False)

Invalid faceOffWinPercentage: 0


In [327]:
for c in ["HoA", "won", "settled_in", "startRinkSide"]:
    print(f"\nValues for {c}:")
    
    game_teams_stats_clean_df \
        .groupBy(c) \
        .count() \
        .orderBy(c) \
        .show(truncate=False)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 287, Finished, Available, Finished, False)


Values for HoA:
+----+-----+
|HoA |count|
+----+-----+
|away|23721|
|home|23721|
+----+-----+


Values for won:
+-----+-----+
|won  |count|
+-----+-----+
|false|24349|
|true |23093|
+-----+-----+


Values for settled_in:
+----------+-----+
|settled_in|count|
+----------+-----+
|OT        |11126|
|REG       |36306|
|tbc       |10   |
+----------+-----+


Values for startRinkSide:
+-------------+-----+
|startRinkSide|count|
+-------------+-----+
|NULL         |2170 |
|left         |24234|
|right        |21038|
+-------------+-----+



In [328]:
invalid_powerplay = (
    game_teams_stats_clean_df
    .filter(
        col("powerPlayGoals") >
        col("powerPlayOpportunities")
    )
)

print(
    "PowerPlayGoals > PowerPlayOpportunities:",
    invalid_powerplay.count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 288, Finished, Available, Finished, False)

PowerPlayGoals > PowerPlayOpportunities: 0


In [329]:
teams_per_game = (
    game_teams_stats_clean_df
    .groupBy("game_id")
    .count()
)

display(
    teams_per_game
    .groupBy("count")
    .count()
    .orderBy("count")
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 289, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a97c6f71-a0a9-47cc-ab7a-5189b2b5acb3)

In [330]:
game_teams_stats_clean_df.printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 290, Finished, Available, Finished, False)

root
 |-- game_id: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- HoA: string (nullable = true)
 |-- won: boolean (nullable = true)
 |-- settled_in: string (nullable = true)
 |-- head_coach: string (nullable = true)
 |-- goals: integer (nullable = true)
 |-- shots: integer (nullable = true)
 |-- hits: integer (nullable = true)
 |-- pim: integer (nullable = true)
 |-- powerPlayOpportunities: integer (nullable = true)
 |-- powerPlayGoals: integer (nullable = true)
 |-- faceOffWinPercentage: double (nullable = true)
 |-- giveaways: integer (nullable = true)
 |-- takeaways: integer (nullable = true)
 |-- blocked: integer (nullable = true)
 |-- startRinkSide: string (nullable = true)



In [334]:
# Rename columns
game_teams_stats_clean_df = game_teams_stats_clean_df.withColumnsRenamed({
                                                    "HoA": "home_or_away",
                                                    "pim": "penalty_in_minutes",
                                                    "powerPlayOpportunities": "power_play_opportunities",
                                                    "powerPlayGoals": "power_play_goals",
                                                    "faceOffWinPercentage": "face_off_win_percentage",
                                                    "startRinkSide": "start_rink_side"})

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 291, Finished, Available, Finished, False)

In [335]:
# Relational checks across columns to ensure they are giving the correct values with reference from the other columns
relational_check(game_teams_stats_clean_df)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 292, Finished, Available, Finished, False)

INFO: Starting relational checks on shots, goals, power_play_goals, power_play_opportunities
INFO: 0 record goals < shots, check passed
INFO: 0 record power_play_goals < goals, check passed
INFO: 0 record showing power_play_opportunities = 0 & goals > 0, check passed


True

In [336]:
# Check if there are any false win/lose or ties scenario.
won_column_validation(game_teams_stats_clean_df)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 293, Finished, Available, Finished, False)

INFO: Checking won column validity


False

In [337]:
# Won column is invalid because there are ties scenarios but won column data type is boolean taking in only true/false scenarios
# create new column game_result and add tie/win/lose result
gts_game_result = create_game_result(game_teams_stats_clean_df)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 294, Finished, Available, Finished, False)

INFO: Creating new column game_result
INFO: Successfully created new column game_result


In [ ]:
gts_game_result.show(10)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 295, Finished, Available, Finished, False)

+----------+-------+------------+-----+----------+--------------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|   game_id|team_id|home_or_away|  won|settled_in|    head_coach|goals|shots|hits|penalty_in_minutes|power_play_opportunities|power_play_goals|face_off_win_percentage|giveaways|takeaways|blocked|start_rink_side|game_result|
+----------+-------+------------+-----+----------+--------------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|2000020001|     21|        away|false|        OT|   Bob Hartley|    2|   28|NULL|                20|                      10|               1|                   NULL|     NULL|     NULL|   NULL|           left|        tie|
|2000020001|     25|        home|false|        OT| Ken Hitchcock|    2|   21|NULL|                22|   

In [338]:
# Checking combination of game_id, home_or_away, goals leads to the correct game_result value.
game_result_validation(gts_game_result)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 296, Finished, Available, Finished, False)

INFO: Checking game_result column validity
INFO: Pass - There are 0 null in game_result.
INFO: Pass - There are 0 false win/lose/ties count.


True

In [339]:
# no issue with game_result
# set final clean data to clean_gts
game_teams_stats_clean_df = gts_game_result
game_teams_stats_clean_df.show(10)
game_teams_stats_clean_df.count()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 297, Finished, Available, Finished, False)

+----------+-------+------------+-----+----------+--------------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|   game_id|team_id|home_or_away|  won|settled_in|    head_coach|goals|shots|hits|penalty_in_minutes|power_play_opportunities|power_play_goals|face_off_win_percentage|giveaways|takeaways|blocked|start_rink_side|game_result|
+----------+-------+------------+-----+----------+--------------+-----+-----+----+------------------+------------------------+----------------+-----------------------+---------+---------+-------+---------------+-----------+
|2000020001|     21|        away|false|        OT|   Bob Hartley|    2|   28|NULL|                20|                      10|               1|                   NULL|     NULL|     NULL|   NULL|           left|        tie|
|2000020001|     25|        home|false|        OT| Ken Hitchcock|    2|   21|NULL|                22|   

47442

### **Load as delta table**

In [340]:
game_teams_stats_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_game_teams_stats")

StatementMeta(, 5a5641b6-3a20-4785-b67e-b70a7ccba1a5, 426, Finished, Available, Finished, False)

In [341]:
silver_game_teams_stats_df = spark.table(
    "silver_game_teams_stats"
)

print(
    "Silver rows:",
    silver_game_teams_stats_df.count()
)

print(
    "Silver columns:",
    len(silver_game_teams_stats_df.columns)
)

silver_game_teams_stats_df.printSchema()

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 298, Finished, Available, Finished, False)

Silver rows: 47442
Silver columns: 18
root
 |-- game_id: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- home_or_away: string (nullable = true)
 |-- won: boolean (nullable = true)
 |-- settled_in: string (nullable = true)
 |-- head_coach: string (nullable = true)
 |-- goals: integer (nullable = true)
 |-- shots: integer (nullable = true)
 |-- hits: integer (nullable = true)
 |-- penalty_in_minutes: integer (nullable = true)
 |-- power_play_opportunities: integer (nullable = true)
 |-- power_play_goals: integer (nullable = true)
 |-- face_off_win_percentage: double (nullable = true)
 |-- giveaways: integer (nullable = true)
 |-- takeaways: integer (nullable = true)
 |-- blocked: integer (nullable = true)
 |-- start_rink_side: string (nullable = true)
 |-- game_result: string (nullable = true)



In [342]:
print(
    "Unique PK:",
    silver_game_teams_stats_df
    .select("game_id", "team_id")
    .distinct()
    .count()
)

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 299, Finished, Available, Finished, False)

Unique PK: 47442


In [345]:
expected_schema = StructType([StructField("game_id", IntegerType()), 
                     StructField("team_id", IntegerType()), 
                     StructField("home_or_away", StringType()), 
                     StructField("won", BooleanType()), 
                     StructField("settled_in", StringType()), 
                     StructField("head_coach", StringType()), 
                     StructField("goals", IntegerType()), 
                     StructField("shots", IntegerType()), 
                     StructField("hits", IntegerType()), 
                     StructField("penalty_in_minutes", IntegerType()), 
                     StructField("power_play_opportunities", IntegerType()), 
                     StructField("power_play_goals", IntegerType()), 
                     StructField("face_off_win_percentage", DoubleType()), 
                     StructField("giveaways", IntegerType()), 
                     StructField("takeaways", IntegerType()), 
                     StructField("blocked", IntegerType()), 
                     StructField("start_rink_side", StringType()),
                     StructField("game_result", StringType()),
                     ])

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 300, Finished, Available, Finished, False)

In [347]:
# Confirm silver_game_teams_stats table exists in directory and silver table is as expected.
# Ensure important columns do not contain null/na.

assert spark.catalog.tableExists("silver_game_teams_stats"), "silver_game_teams_stats table does not exist"
assert silver_game_teams_stats_df.count() > 0, "Silver table is empty"
assert len(silver_game_teams_stats_df.columns) == 18, "Unexpected number of columns"
assert silver_game_teams_stats_df.schema == expected_schema, "Silver table schema does not match expected schema"
assert silver_game_teams_stats_df.count() == silver_game_teams_stats_df.dropDuplicates().count(), "Silver table contains duplicate rows"
assert silver_game_teams_stats_df.filter(col("game_id").isNull()).count() == 0, "game_id contains NULL values"
assert silver_game_teams_stats_df.filter(col("team_id").isNull()).count() == 0, "team_id contains NULL values"
assert silver_game_teams_stats_df.filter(col("home_or_away").isNull()).count() == 0, "home_or_away contains NULL values"
assert silver_game_teams_stats_df.filter(col("goals").isNull()).count() == 0, "goals contains NULL values"
assert silver_game_teams_stats_df.filter(col("game_result").isNull()).count() == 0, "game_result contains NULL values"
assert relational_check(silver_game_teams_stats_df), "Relational data-quality checks failed"
assert game_result_validation(silver_game_teams_stats_df), "game_result validation failed"

StatementMeta(, 563f3625-18b6-472e-ba74-0bf2ddd5a5ea, 301, Finished, Available, Finished, False)

INFO: Starting relational checks on shots, goals, power_play_goals, power_play_opportunities
INFO: 0 record goals < shots, check passed
INFO: 0 record power_play_goals < goals, check passed
INFO: 0 record showing power_play_opportunities = 0 & goals > 0, check passed
INFO: Checking game_result column validity
INFO: Pass - There are 0 null in game_result.
INFO: Pass - There are 0 false win/lose/ties count.
